In [ ]:
import os, gc, json, math, time, warnings
import numpy as np, pandas as pd, torch, timm, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
warnings.filterwarnings("ignore")
dev = "cuda"

OUT = r"G:\runs"; VW = os.path.join(OUT, "vinbig"); VT = os.path.join(OUT, "vit")
FIG = os.path.join(VT, "figures"); os.makedirs(FIG, exist_ok=True)
ARCH = "vit_small_patch16_224.augreg_in21k_ft_in1k"
EPOCHS, SEEDS, BS, LR, WD = 7, [0, 1, 2], 64, 5e-5, 0.05
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5 = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]; L = 6
TS = ["A", "B", "C", "A+B", "A+C", "B+C"]            # the 6 training sets
SITE = {"A": "NIH", "B": "CheXpert", "C": "VinBigData"}

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".pdf"), bbox_inches="tight"); fig.savefig(os.path.join(FIG, name + ".png"), dpi=600, bbox_inches="tight"); plt.show()
print(torch.cuda.get_device_name(0), "| figures ->", FIG)

In [ ]:
nih = pd.read_csv(r"G:\nih_split.csv"); off = nih.official.values == "test"; te_i = np.where(off)[0]
tr_i = np.where(nih.split_rev.values == "train")[0]; va_i = np.where(nih.split_rev.values == "val")[0]
NIH_A = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224)); Yn = nih[LABELS6].values.astype(int)

cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True)
cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
CX_A = np.memmap(r"G:\cx_trainval_224.u8", dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224)); CX_T = np.load(r"G:\cx_test_224.npy")
Ycm = cxm[LABELS6].values.astype(int); Yc = cxt[LABELS6].values.astype(int)

vb = pd.read_csv(os.path.join(VW, "vinbig_image_labels.csv")); NVB = len(vb)
VB_A = np.memmap(os.path.join(VW, "vb_train_224.u8"), dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
assert len(json.load(open(os.path.join(VW, "vb_train_progress.json")))["bad"]) == 0
Yvb = np.zeros((NVB, L), int)
for f in FIND5: Yvb[:, LABELS6.index(f)] = vb[f].values
VALID_C = np.array([l != "Edema" for l in LABELS6])                    # VinBigData has no Edema -> masked everywhere

sp = os.path.join(OUT, "fair_ens", "vinbig_split.csv")                  # reuse the split of full11 if it exists
if os.path.exists(sp):
    s_ = pd.read_csv(sp); vb_va = np.where(s_.split.values == "val")[0]; vb_tr = np.where(s_.split.values == "train")[0]
else:
    perm = np.random.default_rng(0).permutation(NVB); vb_va = np.sort(perm[:NVB // 5]); vb_tr = np.sort(perm[NVB // 5:])

SRC = {"A": dict(arr=NIH_A, Y=Yn,  tr=tr_i,  va=va_i,  valid=np.ones(L, bool)),
       "B": dict(arr=CX_A,  Y=Ycm, tr=cx_tr, va=cx_va, valid=np.ones(L, bool)),
       "C": dict(arr=VB_A,  Y=Yvb, tr=vb_tr, va=vb_va, valid=VALID_C)}
TEST = {"A": (NIH_A, te_i), "B": (CX_T, np.arange(len(CX_T))), "C": (VB_A, np.arange(NVB))}   # C is only a TEST site when it is not in the training set
YT = {"A": Yn[te_i], "B": Yc, "C": Yvb}; VALID = {"A": np.ones(L, bool), "B": np.ones(L, bool), "C": VALID_C}
VAL_Y = {"A": Yn[va_i], "B": Ycm[cx_va], "C": Yvb[vb_va]}
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
GRP = {"A": groups(nih.patient_id.values[off]), "B": groups(cxt.patient_id.values), "C": None}   # patient bootstrap; VinBigData has no patient id -> image bootstrap
def resample(T, rng):
    if GRP[T] is None: n = len(YT[T]); return rng.integers(0, n, n)
    g = GRP[T]; return np.concatenate([g[k] for k in rng.integers(0, len(g), len(g))])
print({k: (len(v["tr"]), len(v["va"])) for k, v in SRC.items()}, "| test:", len(te_i), len(CX_T), NVB)

In [ ]:
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def lab_auc(Y, P, valid):                                   # per-label AUROC, NaN where the label is absent or too rare
    return np.array([fauc(Y[:, j], P[:, j]) if valid[j] and 5 <= Y[:, j].sum() <= len(Y) - 5 else np.nan for j in range(L)])
def mauc(Y, P, valid): return float(np.nanmean(lab_auc(Y, P, valid)))
def mauprc(Y, P, valid): return float(np.mean([average_precision_score(Y[:, j], P[:, j]) for j in range(L) if valid[j] and Y[:, j].sum() >= 5]))
sig = lambda z: 1 / (1 + np.exp(-z))
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
GRID = np.linspace(0.02, 0.95, 94)
def best_thr(y, p):
    pr = p[:, None] >= GRID[None, :]; y1 = (y == 1)[:, None]; tp = (pr & y1).sum(0); fp = pr.sum(0) - tp; fn = y1.sum() - tp
    return GRID[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))]
def f1_at(y, p, t):
    pr = p >= t; y1 = y == 1; tp = (pr & y1).sum(); d = 2 * tp + (pr & ~y1).sum() + (~pr & y1).sum(); return 2 * tp / d if d else 0.0
def ece_of(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.any(): e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e

def make_model(pretrained=True): return timm.create_model(ARCH, pretrained=pretrained, num_classes=L, drop_path_rate=0.1).to(dev)
_tmp = make_model(True); CFG = _tmp.pretrained_cfg; del _tmp
MEAN = torch.tensor(CFG["mean"], device=dev).view(1, 3, 1, 1); STD = torch.tensor(CFG["std"], device=dev).view(1, 3, 1, 1)   # ViT augreg uses 0.5/0.5, not ImageNet stats
def gpu_batch(A, idx, flip=False):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device=dev) < 0.5; x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD
@torch.inference_mode()
def predict(m, A, idx, bs=128):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda"): o = m(gpu_batch(A, idx[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()

def batches(srcs, seed, ep):                                # natural pooled sampling (the best DG baseline in your earlier runs)
    rng = np.random.default_rng(seed * 100 + ep)
    dom = np.concatenate([np.full(len(SRC[k]["tr"]), i) for i, k in enumerate(srcs)]); pos = np.concatenate([SRC[k]["tr"] for k in srcs])
    perm = rng.permutation(len(pos))
    for k in range(math.ceil(len(pos) / BS)):
        s = perm[k * BS:(k + 1) * BS]
        yield {srcs[i]: np.sort(pos[s][dom[s] == i]) for i in range(len(srcs)) if (dom[s] == i).any()}
def val_scores(m, srcs):                                    # SOURCE validation only, never the target
    o = {}
    for k in srcs:
        P = predict(m, SRC[k]["arr"], SRC[k]["va"]); o[k] = mauc(SRC[k]["Y"][SRC[k]["va"]], P, SRC[k]["valid"])
    return o
def base_path(ts, s): return os.path.join(VT, f"vit_{ts.replace('+', 'p')}_seed{s}")

def train_run(ts, seed):
    srcs = ts.split("+"); b = base_path(ts, seed); BEST, LAST, HIST, PRD = b + ".pt", b + "_last.pt", b + "_history.csv", b + "_preds.npz"
    if os.path.exists(PRD): print("skip (finished):", os.path.basename(b)); return
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model(); opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); total, warm = EPOCHS * spe, min(300, spe)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda"); hist, best, start = [], 0.0, 0
    if os.path.exists(LAST):
        ck = torch.load(LAST, map_location=dev, weights_only=False)
        m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1; print("resumed, next epoch", start + 1)
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev); n = 0
        for batch in tqdm(batches(srcs, seed, ep), total=spe, leave=False, desc=f"ViT {ts} seed {seed} ep {ep + 1}/{EPOCHS}"):
            xs, ys, ms = [], [], []
            for k, idx in batch.items():
                xs.append(gpu_batch(SRC[k]["arr"], idx, flip=True)); ys.append(SRC[k]["Y"][idx]); ms.append(np.tile(SRC[k]["valid"], (len(idx), 1)))
            x = torch.cat(xs); y = torch.from_numpy(np.concatenate(ys)).to(dev).float(); mk = torch.from_numpy(np.concatenate(ms)).to(dev).float()
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda"): z = m(x)
            loss = (F.binary_cross_entropy_with_logits(z.float(), y, reduction="none") * mk).sum() / mk.sum()
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach() * len(x); n += len(x)
        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))
        row = dict(epoch=ep + 1, train_loss=float(tl) / n, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"ViT {ts} seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f} | val AUROC {vs} | {row['sec'] / 60:.1f} min")
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep), LAST)
    m.load_state_dict(torch.load(BEST, map_location=dev, weights_only=True)); z = {}
    z["A_test_P"] = predict(m, NIH_A, te_i); z["B_test_P"] = predict(m, CX_T, np.arange(len(CX_T)))
    if "C" not in srcs: z["C_test_P"] = predict(m, VB_A, np.arange(NVB))        # C is a clean test site only if it was not trained on
    for k in ["A", "B"] + (["C"] if "C" in srcs else []): z[f"{k}_val_P"] = predict(m, SRC[k]["arr"], SRC[k]["va"])   # for thresholds + recalibration pools
    np.savez(PRD, **z); del m, opt, sch, scaler; gc.collect(); torch.cuda.empty_cache()

In [ ]:
SEEDS_RUN = SEEDS          # tip: first run [0] to test the pipeline end to end, then [0, 1, 2]
for ts in TS:
    for s in SEEDS_RUN: train_run(ts, s)

In [ ]:
rows = []
for ts in ["C", "A+C", "B+C"]:
    v = [mauc(Yvb[vb_va], P_[(ts, s)]["C_val_P"], VALID_C) for s in seeds_of(ts)]
    rows.append(dict(train=ts, test="C (val split ‡)", auroc=f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}"))
display(pd.DataFrame(rows))

In [ ]:
P_ = {(ts, s): dict(np.load(base_path(ts, s) + "_preds.npz")) for ts in TS for s in SEEDS if os.path.exists(base_path(ts, s) + "_preds.npz")} 
seeds_of = lambda ts: [s for s in SEEDS if (ts, s) in P_]
CELLS = [(ts, T) for ts in TS for T in "ABC" if not (T == "C" and "C" in ts.split("+"))]
PM = {(ts, T): np.mean([P_[(ts, s)][f"{T}_test_P"] for s in seeds_of(ts)], 0) for ts, T in CELLS}      # seed-averaged probabilities
seen = lambda ts, T: T in ts.split("+")

rows = []
for ts, T in CELLS:
    a = [mauc(YT[T], P_[(ts, s)][f"{T}_test_P"], VALID[T]) for s in seeds_of(ts)]
    p = [mauprc(YT[T], P_[(ts, s)][f"{T}_test_P"], VALID[T]) for s in seeds_of(ts)]
    rows.append(dict(train=ts, test=T, seen=seen(ts, T), auroc=np.mean(a), auroc_sd=np.std(a, ddof=1) if len(a) > 1 else 0, auprc=np.mean(p), auprc_sd=np.std(p, ddof=1) if len(p) > 1 else 0))
RES = pd.DataFrame(rows); RES.to_csv(os.path.join(VT, "table_vit_main.csv"), index=False)
fmt = lambda r, m: f"{r[m]:.3f} ± {r[m + '_sd']:.3f}" + (" †" if r.seen else "")
for m in ["auroc", "auprc"]:
    t = RES.assign(v=RES.apply(lambda r: fmt(r, m), axis=1)).pivot(index="train", columns="test", values="v").reindex(TS)
    print(f"\n=== ViT-S, {m.upper()} (mean ± sd over seeds; † = test site was in the training set; column C excludes Edema) ==="); display(t)

In [ ]:
NB = 300; rng = np.random.default_rng(0)
BSL = {c: np.zeros((NB, L)) for c in CELLS}                   # per-label AUROC per resample
for r in tqdm(range(NB), desc="bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}                 # one resample per site, shared by all models = paired
    for ts, T in CELLS: BSL[(ts, T)][r] = lab_auc(YT[T][ix[T]], PM[(ts, T)][ix[T]], VALID[T])
BSM = {c: np.nanmean(v, 1) for c, v in BSL.items()}
PTL = {c: lab_auc(YT[c[1]], PM[c], VALID[c[1]]) for c in CELLS}; PTM = {c: np.nanmean(v) for c, v in PTL.items()}
ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)

rows = [dict(train=ts, test=T, auroc=PTM[(ts, T)], lo=ci(BSM[(ts, T)])[0], hi=ci(BSM[(ts, T)])[1], seen=seen(ts, T)) for ts, T in CELLS]
CI = pd.DataFrame(rows); CI.to_csv(os.path.join(VT, "table_vit_auroc_ci.csv"), index=False); display(CI.round(3))

gap = {"A→B (NIH→CheXpert)": ("A", "B"), "B→A (CheXpert→NIH)": ("B", "A")}      # gap = in-hospital AUROC − external AUROC, 6 labels
GB, GP = {}, {}
for nm, (s_, t_) in gap.items():
    GB[nm] = BSL[(s_, s_)] - BSL[(s_, t_)]; GP[nm] = PTL[(s_, s_)] - PTL[(s_, t_)]
k1, k2 = list(gap)
asym_b = np.nanmean(GB[k1], 1) - np.nanmean(GB[k2], 1); asym_p = np.nanmean(GP[k1]) - np.nanmean(GP[k2])
rows = [dict(quantity=k, mean_gap=np.nanmean(GP[k]), lo=np.percentile(np.nanmean(GB[k], 1), 2.5), hi=np.percentile(np.nanmean(GB[k], 1), 97.5)) for k in gap]
rows.append(dict(quantity="Asymmetry (A→B gap − B→A gap)", mean_gap=asym_p, lo=np.percentile(asym_b, 2.5), hi=np.percentile(asym_b, 97.5)))
GAPT = pd.DataFrame(rows); display(GAPT.round(3)); GAPT.to_csv(os.path.join(VT, "table_vit_gaps.csv"), index=False)
print("Asymmetry CI excludes 0 ->", not (GAPT.lo.iloc[-1] <= 0 <= GAPT.hi.iloc[-1]))

In [ ]:
# Fig 1: training curves
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
for ts in TS:
    for s in seeds_of(ts):
        h = pd.read_csv(base_path(ts, s) + "_history.csv"); c = f"C{TS.index(ts)}"
        ax[0].plot(h.epoch, h.train_loss, color=c, alpha=.6, label=ts if s == seeds_of(ts)[0] else None); ax[1].plot(h.epoch, h.val_mean, color=c, alpha=.6)
ax[0].set(xlabel="Epoch", ylabel="Train loss", title="Training loss"); ax[1].set(xlabel="Epoch", ylabel="Source-val AUROC", title="Validation AUROC (source only)")
ax[0].legend(frameon=False, ncol=2); [a.grid(True, alpha=.3) for a in ax]; fig.tight_layout(); save_fig(fig, "fig_vit_training_curves")

# Fig 2: transfer matrix (rows = training set, columns = test site)
M = np.array([[PTM.get((ts, T), np.nan) for T in "ABC"] for ts in TS])
fig, ax = plt.subplots(figsize=(COL1 * 1.3, 3.0)); im = ax.imshow(M, cmap="Greys", vmin=np.nanmin(M) - .03, vmax=np.nanmax(M) + .02)
for i, ts in enumerate(TS):
    for j, T in enumerate("ABC"):
        if np.isnan(M[i, j]): ax.text(j, i, "–", ha="center", va="center"); continue
        ax.text(j, i, f"{M[i, j]:.3f}" + ("†" if seen(ts, T) else ""), ha="center", va="center", color="white" if M[i, j] > np.nanmean(M) else "black")
ax.set_xticks(range(3)); ax.set_xticklabels([f"{SITE[t]}\n({t})" + ("\n5 findings" if t == "C" else "") for t in "ABC"]); ax.set_yticks(range(len(TS))); ax.set_yticklabels(TS)
ax.set(xlabel="Tested on", ylabel="Trained on", title="ViT-S AUROC († = site seen in training)"); ax.grid(False); fig.tight_layout(); save_fig(fig, "fig_vit_transfer_matrix")

# Fig 3: AUROC with 95% CI for every external direction
ext = [(ts, T) for ts, T in CELLS if not seen(ts, T)]; xs = np.arange(len(ext))
fig, ax = plt.subplots(figsize=(COL2, 2.9)); v = np.array([PTM[c] for c in ext]); c_ = np.array([ci(BSM[c]) for c in ext])
ax.bar(xs, v, 0.65, yerr=[np.clip(v - c_[:, 0], 0, None), np.clip(c_[:, 1] - v, 0, None)], capsize=2, color=["0.3" if "+" not in a else "0.7" for a, _ in ext], edgecolor="k", lw=.4)
ax.set_xticks(xs); ax.set_xticklabels([f"{a}→{b}" for a, b in ext], rotation=40, ha="right"); ax.set_ylim(max(0.5, v.min() - .08), v.max() + .04)
ax.set(ylabel="Macro AUROC (external site)", title="ViT-S external performance (dark = single source, light = pooled sources)"); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_external_auroc")

# Fig 4: per-label AUROC heatmap, external directions
D = np.array([PTL[c] for c in ext]); fig, ax = plt.subplots(figsize=(COL2, 3.2)); im = ax.imshow(D, cmap="viridis", aspect="auto")
for i in range(D.shape[0]):
    for j in range(L): ax.text(j, i, "–" if np.isnan(D[i, j]) else f"{D[i, j]:.2f}", ha="center", va="center", color="w", fontsize=6)
ax.set_xticks(range(L)); ax.set_xticklabels(LABELS6, rotation=30, ha="right"); ax.set_yticks(range(len(ext))); ax.set_yticklabels([f"{a}→{b}" for a, b in ext]); ax.grid(False)
fig.colorbar(im, ax=ax, label="AUROC"); ax.set_title("Per-disease external AUROC (ViT-S)"); fig.tight_layout(); save_fig(fig, "fig_vit_per_label_heatmap")

# Fig 5: gaps, asymmetry, per-label gaps
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8), gridspec_kw=dict(width_ratios=[1.6, 1])); x = np.arange(L)
for o, (nm, c) in zip([-.2, .2], zip(gap, ["0.35", "0.75"])):
    pt = GP[nm]; cc = ci(GB[nm]); ax[0].bar(x + o, pt, .4, yerr=[np.clip(pt - cc[0], 0, None), np.clip(cc[1] - pt, 0, None)], capsize=1.5, color=c, edgecolor="k", lw=.4, label=nm)
ax[0].axhline(0, color="k", lw=.6); ax[0].set_xticks(x); ax[0].set_xticklabels(LABELS6, rotation=35, ha="right"); ax[0].set(ylabel="AUROC drop (in-hospital − external)"); ax[0].legend(frameon=False); ax[0].grid(True, axis="y", alpha=.3)
g = GAPT; ax[1].bar(range(3), g.mean_gap, .6, yerr=[np.clip(g.mean_gap - g.lo, 0, None), np.clip(g.hi - g.mean_gap, 0, None)], capsize=2, color=["0.35", "0.75", "0.55"], edgecolor="k", lw=.4)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(["A→B", "B→A", "Asym."]); ax[1].axhline(0, color="k", lw=.6); ax[1].set(ylabel="Mean AUROC difference"); ax[1].grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_gaps_asymmetry")

In [ ]:
def thr_for(ts, s):
    thr = np.full(L, 0.5)
    for j in range(L):
        ks = [k for k in ts.split("+") if SRC[k]["valid"][j]]
        if ks: thr[j] = best_thr(np.concatenate([VAL_Y[k][:, j] for k in ks]), np.concatenate([P_[(ts, s)][f"{k}_val_P"][:, j] for k in ks]))
    return thr
rows = []
for ts, T in CELLS:
    for s in seeds_of(ts):
        P = P_[(ts, s)][f"{T}_test_P"]; Y = YT[T]; v = np.where(VALID[T])[0]; thr = thr_for(ts, s)      # source-tuned thresholds, applied unchanged to the target
        rows.append(dict(train=ts, test=T, seed=s, seen=seen(ts, T), ece=np.mean([ece_of(Y[:, j], P[:, j]) for j in v]), f1=np.mean([f1_at(Y[:, j], P[:, j], thr[j]) for j in v])))
CAL = pd.DataFrame(rows); CAL.to_csv(os.path.join(VT, "table_vit_calibration.csv"), index=False)
for m in ["ece", "f1"]:
    t = CAL.groupby(["train", "test"])[m].agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}").unstack("test").reindex(TS)
    print(f"\n=== ViT-S {m.upper()} (source-tuned thresholds) ==="); display(t)

def reliab(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); xs_, ys_, n_ = [], [], []
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 30: xs_.append(p[mk].mean()); ys_.append(y[mk].mean())
    return xs_, ys_
cases = [("A", "A", "A→A (in-hospital)"), ("A", "B", "A→B"), ("B", "B", "B→B (in-hospital)"), ("B", "A", "B→A"), ("A+B", "C", "A+B→C"), ("A+C", "B", "A+C→B")]
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.6), sharey=True)
for a_, grp, ttl in zip(ax, [cases[:2], cases[2:4], cases[4:]], ["Trained on NIH", "Trained on CheXpert", "Pooled models"]):
    for ts, T, nm in grp:
        v = np.where(VALID[T])[0]; P = P_[(ts, 0)][f"{T}_test_P"][:, v].ravel(); Y = YT[T][:, v].ravel()
        xs_, ys_ = reliab(Y, P); a_.plot(xs_, ys_, "-o", ms=3, label=nm)
    a_.plot([0, 1], [0, 1], "k:", lw=.6); a_.set(title=ttl, xlabel="Predicted probability"); a_.legend(frameon=False); a_.grid(True, alpha=.3)
ax[0].set_ylabel("Observed frequency"); fig.tight_layout(); save_fig(fig, "fig_vit_reliability")

t = CAL[~CAL.seen].groupby(["train", "test"])[["ece", "f1"]].mean().reset_index(); lab = [f"{a}→{b}" for a, b in zip(t.train, t.test)]
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
for a_, m, ttl in zip(ax, ["ece", "f1"], ["ECE (lower is better)", "Macro-F1, source threshold (higher is better)"]):
    a_.bar(range(len(t)), t[m], color="0.55", edgecolor="k", lw=.4); a_.set_xticks(range(len(t))); a_.set_xticklabels(lab, rotation=40, ha="right"); a_.set_title(ttl); a_.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_ece_f1_external")

In [ ]:
def fit_platt(z, y):
    lr = LogisticRegression(C=1e6, max_iter=500).fit(z.reshape(-1, 1), y); return float(np.clip(lr.coef_[0, 0], .1, 5)), float(lr.intercept_[0])
def correct(meth, zp, yp, ze, t0):
    a, b, t = 1.0, 0.0, t0; npos = int(yp.sum()); nneg = len(yp) - npos
    if npos >= 5 and nneg >= 5:
        if meth == "Platt + threshold": a, b = fit_platt(zp, yp)
        t = best_thr(yp, sig(a * zp + b))
    return sig(a * ze + b), t
KS, DRAWS, METH = [0, 25, 50, 100, 250, 500, 1000, 2500, 5000], 20, ["Threshold only", "Platt + threshold"]
POOL = {"A": ("B", VAL_Y["B"], "B_val_P"), "B": ("A", VAL_Y["A"], "A_val_P")}        # model trained on A -> pool = CheXpert val; B -> NIH val
rows, ref = [], []
for ts in ["A", "B"]:
    T, Yp, key = POOL[ts]
    for s in seeds_of(ts):
        z = P_[(ts, s)]; Pe, Ye, Pp = z[f"{T}_test_P"], YT[T], z[key]; Ze, Zp = logit(Pe), logit(Pp); thr0 = thr_for(ts, s)
        Pi, Yi = z[f"{ts}_test_P"] if f"{ts}_test_P" in z else None, YT[ts]
        ref.append(dict(ts=ts, seed=s, ece_in=np.mean([ece_of(Yi[:, j], Pi[:, j]) for j in range(L)]), f1_in=np.mean([f1_at(Yi[:, j], Pi[:, j], thr0[j]) for j in range(L)])))
        for K in KS:
            for dr in range(1 if K == 0 else DRAWS):
                idx = np.random.default_rng([s, K, dr]).choice(len(Yp), K, replace=False) if K else np.array([], int)
                for me in METH:
                    es, fs = [], []
                    for j in range(L):
                        pe, t = correct(me, Zp[idx, j], Yp[idx, j], Ze[:, j], thr0[j]) if K else (sig(Ze[:, j]), thr0[j])
                        es.append(ece_of(Ye[:, j], pe)); fs.append(f1_at(Ye[:, j], pe, t))
                    rows.append(dict(ts=ts, seed=s, method=me, K=K, draw=dr, ece=np.mean(es), f1=np.mean(fs)))
RC, RF = pd.DataFrame(rows), pd.DataFrame(ref); RC.to_csv(os.path.join(VT, "recal_raw.csv"), index=False)
ps = RC.groupby(["ts", "method", "K", "seed"])[["ece", "f1"]].mean().reset_index(); refm = RF.groupby("ts")[["ece_in", "f1_in"]].mean()
tab = ps[ps.K.isin([0, 100, 500, 2500])].groupby(["ts", "method", "K"])[["ece", "f1"]].mean().round(3).unstack("K"); display(tab); tab.to_csv(os.path.join(VT, "table_vit_recal.csv"))

fig, ax = plt.subplots(2, 2, figsize=(COL2, 5.0)); DN = {"A": "A→B (NIH→CheXpert)", "B": "B→A (CheXpert→NIH)"}
for r_, ts in enumerate(["A", "B"]):
    for c_, (m, ttl) in enumerate([("ece", "ECE (lower is better)"), ("f1", "Macro-F1 (higher is better)")]):
        for me, col in zip(METH, ["C0", "C3"]):
            g = ps[(ps.ts == ts) & (ps.method == me)].groupby("K")[m].agg(["mean", "std"])
            ax[r_, c_].plot(g.index, g["mean"], "-o", ms=3, color=col, label=me); ax[r_, c_].fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"], color=col, alpha=.15, lw=0)
        ax[r_, c_].axhline(refm.loc[ts, m + "_in"], color="k", ls="--", lw=.7, label="in-hospital reference"); ax[r_, c_].set_xscale("symlog", linthresh=25)
        ax[r_, c_].set(xlabel="k labelled target images", title=f"{DN[ts]}: {ttl}"); ax[r_, c_].grid(True, alpha=.3)
ax[0, 0].legend(frameon=False); fig.tight_layout(); save_fig(fig, "fig_vit_recalibration")


In [ ]:
# ---- E1: seed ensemble vs single seed -----------------------------------------------------------------
rows = []
for ts, T in CELLS:
    if len(seeds_of(ts)) < 2: continue
    sg = [mauc(YT[T], P_[(ts, s)][f"{T}_test_P"], VALID[T]) for s in seeds_of(ts)]
    rows.append(dict(direction=f"{ts}→{T}", single_mean=np.mean(sg), seed_ensemble=PTM[(ts, T)], gain=PTM[(ts, T)] - np.mean(sg)))
E1 = pd.DataFrame(rows); display(E1.round(4)); E1.to_csv(os.path.join(VT, "table_ens_seed.csv"), index=False)

# ---- E2: cross-source ensemble vs pooled training (equal member count) --------------------------------------
TGT = {"A": (["B", "C"], "B+C"), "B": (["A", "C"], "A+C"), "C": (["A", "B"], "A+B")}
S3 = [s for s in SEEDS if all((ts, s) in P_ for ts in TS)]; assert len(S3) >= 2, "need >= 2 complete seeds"
def members(T):
    srcs, pool = TGT[T]
    cross = [np.mean([P_[(k, s)][f"{T}_test_P"] for k in srcs], 0) for s in S3]                                  # 2 single-source models per seed
    pool1 = [P_[(pool, s)][f"{T}_test_P"] for s in S3]                                                           # 1 pooled model per seed
    pool2 = [np.mean([P_[(pool, s)][f"{T}_test_P"], P_[(pool, S3[(i + 1) % len(S3)])][f"{T}_test_P"]], 0) for i, s in enumerate(S3)]   # 2 pooled models = same inference cost
    return dict(cross=cross, pool1=pool1, pool2=pool2)
MB = {T: members(T) for T in "ABC"}; rng = np.random.default_rng(1); NB2 = 300
ens_auc = lambda T, ix, k: np.mean([mauc(YT[T][ix], p[ix], VALID[T]) for p in MB[T][k]])
pt = {(T, k): ens_auc(T, np.arange(len(YT[T])), k) for T in "ABC" for k in ["cross", "pool1", "pool2"]}
bs = {(T, k): np.zeros(NB2) for T in "ABC" for k in ["cross", "pool1", "pool2"]}
for r in tqdm(range(NB2), desc="ensemble bootstrap"):
    for T in "ABC":
        ix = resample(T, rng)
        for k in ["cross", "pool1", "pool2"]: bs[(T, k)][r] = ens_auc(T, ix, k)
rows = []
for T in "ABC":
    for ref_, nm in [("pool1", "pooled ×1"), ("pool2", "pooled ×2 (equal cost)")]:
        d = bs[(T, "cross")] - bs[(T, ref_)]; p = min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / NB2)
        rows.append(dict(target=f"{SITE[T]} ({T})", comparison=f"cross-source ensemble − {nm}", diff=pt[(T, "cross")] - pt[(T, ref_)], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=p))
E2 = pd.DataFrame(rows); o = np.argsort(E2.p.values); holm = np.empty(len(E2)); run = 0
for r_, i in enumerate(o): run = max(run, min(1, (len(E2) - r_) * E2.p.values[i])); holm[i] = run
E2["p_holm"] = holm; display(E2.round(4)); E2.to_csv(os.path.join(VT, "table_ens_cross_source.csv"), index=False)

fig, ax = plt.subplots(figsize=(COL2, 2.9)); w = .26
for i, (k, nm, c) in enumerate([("pool1", "Pooled ×1", "0.75"), ("pool2", "Pooled ×2", "0.5"), ("cross", "Cross-source ensemble", "0.2")]):
    v = np.array([pt[(T, k)] for T in "ABC"]); cc = np.array([np.percentile(bs[(T, k)], [2.5, 97.5]) for T in "ABC"])
    ax.bar(np.arange(3) + (i - 1) * w, v, w, yerr=[np.clip(v - cc[:, 0], 0, None), np.clip(cc[:, 1] - v, 0, None)], capsize=2, color=c, edgecolor="k", lw=.4, label=nm)
ax.set_xticks(range(3)); ax.set_xticklabels([f"Test {SITE[T]}\n(sources: {'+'.join(TGT[T][0])})" for T in "ABC"]); ax.set_ylim(.6, None); ax.set(ylabel="Macro AUROC"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_ensemble_cross_source")

In [ ]:
class ViTCAM:
    def __init__(self, m):
        self.m = m; self.np_ = m.num_prefix_tokens; m.blocks[-1].norm1.register_forward_hook(self._fwd)
    def _fwd(self, mod, i, out):
        self.act = out
        if out.requires_grad: out.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, x, j):                                   # j: int or LongTensor [B] (label per image)
        self.m.zero_grad(set_to_none=True)
        with torch.enable_grad():
            z = self.m(x); j = torch.full((len(x),), j, device=x.device) if isinstance(j, int) else j
            z.gather(1, j.view(-1, 1)).sum().backward()
        a, g = self.act[:, self.np_:].detach(), self.grad[:, self.np_:].detach()
        cam = torch.relu((g.mean(1, keepdim=True) * a).sum(-1)); n = int(cam.shape[1] ** .5)
        return F.interpolate(cam.view(-1, 1, n, n), size=224, mode="bilinear", align_corners=False)[:, 0].cpu().numpy(), torch.sigmoid(z.detach()).cpu().numpy()
def load_vit(ts, s):
    m = make_model(False); m.load_state_dict(torch.load(base_path(ts, s) + ".pt", map_location=dev, weights_only=True)); return m.eval()
def cam_batch(cam, A, idx, j):
    x = gpu_batch(A, idx); return cam(x, j)
def norm_cam(c): return c / (c.max() + 1e-8)

# --- Fig: TP / FP / FN examples, 3 external directions x 2 diseases ---------------------------------------------
CASES = [("A", "B", "A→B (NIH model on CheXpert)"), ("B", "A", "B→A (CheXpert model on NIH)"), ("A+B", "C", "A+B→C (pooled model on VinBigData)")]
for ts, T, ttl in CASES:
    m = load_vit(ts, 0); cam = ViTCAM(m); A_, idxs = TEST[T]; P = P_[(ts, 0)][f"{T}_test_P"]; Y = YT[T]
    for lab in ["Cardiomegaly", "Pleural Effusion"]:
        j = LABELS6.index(lab); p, y = P[:, j], Y[:, j]
        pick = {"True positive": np.where(y == 1)[0][np.argsort(-p[y == 1])[:4]], "False positive": np.where(y == 0)[0][np.argsort(-p[y == 0])[:4]], "False negative": np.where(y == 1)[0][np.argsort(p[y == 1])[:4]]}
        fig, ax = plt.subplots(3, 4, figsize=(COL2, 5.4))
        for r_, (nm, ii) in enumerate(pick.items()):
            cams, probs = cam_batch(cam, A_, idxs[ii], j)
            for c_ in range(4):
                a_ = ax[r_, c_]; a_.imshow(A_[idxs[ii[c_]]], cmap="gray"); a_.imshow(norm_cam(cams[c_]), cmap="jet", alpha=.4); a_.axis("off")
                a_.set_title(f"{nm}\np={probs[c_, j]:.2f}", fontsize=6)
        fig.suptitle(f"{ttl} · {lab}", fontsize=8); fig.tight_layout(); save_fig(fig, f"fig_vit_gradcam_{ts.replace('+', 'p')}_to_{T}_{lab.replace(' ', '')}")
    del m, cam; torch.cuda.empty_cache()

# --- Quantitative: share of CAM mass in the outer 10% border (shortcut check) ----------------------------------
def border_share(c, f=.1):
    k = int(224 * f); m_ = np.ones((224, 224), bool); m_[k:-k, k:-k] = False; return float(c[m_].sum() / (c.sum() + 1e-8))
rng = np.random.default_rng(0); rows = []
for ts, T, ttl in CASES:
    m = load_vit(ts, 0); cam = ViTCAM(m); A_, idxs = TEST[T]; P = P_[(ts, 0)][f"{T}_test_P"]
    sel = np.sort(rng.choice(len(idxs), 300, replace=False))
    for i in range(0, len(sel), 32):
        ii = sel[i:i + 32]; j = torch.from_numpy(np.where(VALID[T][None], P[ii], -1).argmax(1)).to(dev)             # CAM of each image's top-scored finding
        cams, _ = cam_batch(cam, A_, idxs[ii], j); rows += [dict(case=ttl.split(" (")[0], border=border_share(c)) for c in cams]
    del m, cam; torch.cuda.empty_cache()
BR = pd.DataFrame(rows); print("uniform CAM would put 36% of its mass in the border (outer 10% frame)"); display(BR.groupby("case").border.agg(["mean", "std"]).round(3)); BR.to_csv(os.path.join(VT, "table_gradcam_border.csv"), index=False)
fig, ax = plt.subplots(figsize=(COL1 * 1.4, 2.6)); cs = list(BR.case.unique()); ax.boxplot([BR[BR.case == c].border for c in cs], labels=cs, showfliers=False)
ax.axhline(.36, color="r", ls="--", lw=.7, label="uniform CAM"); ax.set(ylabel="CAM mass in outer 10% border"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_gradcam_border")

In [ ]:
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score
m = load_vit("A+B", 0); rng = np.random.default_rng(0); feats, site = [], []
@torch.inference_mode()
def embed(A, idx):
    out = []
    for i in range(0, len(idx), 128):
        with torch.autocast("cuda"): out.append(m.forward_head(m.forward_features(gpu_batch(A, idx[i:i + 128])), pre_logits=True).float().cpu())
    return torch.cat(out).numpy()
for T in "ABC":
    A_, idxs = TEST[T]; ii = np.sort(rng.choice(len(idxs), 600, replace=False)); feats.append(embed(A_, idxs[ii])); site += [T] * 600
X = np.concatenate(feats); site = np.array(site); Z2 = TSNE(2, perplexity=35, random_state=0, init="pca").fit_transform(X)
acc = cross_val_score(LogisticRegression(max_iter=2000), X, site, cv=5).mean()
fig, ax = plt.subplots(figsize=(COL1 * 1.5, 3.2))
for T, c in zip("ABC", ["C0", "C1", "C2"]): ax.scatter(*Z2[site == T].T, s=4, alpha=.6, color=c, label=f"{SITE[T]} ({T})")
ax.set(title=f"ViT-S (A+B) embeddings by hospital · site-probe accuracy {acc:.2f} (chance 0.33)", xticks=[], yticks=[]); ax.legend(frameon=False, markerscale=3)
fig.tight_layout(); save_fig(fig, "fig_vit_tsne_site"); print("site-probe accuracy:", round(acc, 3)); del m; torch.cuda.empty_cache()

In [ ]:
import shutil
shutil.make_archive(os.path.join(OUT, "VIT_bundle"), "zip", VT); print("bundle:", os.path.join(OUT, "VIT_bundle.zip"))
print(sorted(f for f in os.listdir(FIG) if f.endswith(".png")))

In [ ]:
METHS = ["ERM (natural)", "ERM (balanced)", "DANN", "CORAL", "GroupDRO"]
TAG = {"ERM (natural)": "", "ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral", "GroupDRO": "gdro"}
POOLED = ["A+B", "A+C", "B+C"]; UNSEEN = {"A+B": "C", "A+C": "B", "B+C": "A"}
def mpath(ts, meth, s): return base_path(ts, s) + ("_" + TAG[meth] if TAG[meth] else "")   # ERM (natural) = the models from Cell 3

def batches_bal(srcs, seed, ep):                         # equal number of images from every hospital in every step
    rng = np.random.default_rng(seed * 100 + ep); h = BS // len(srcs)
    for _ in range(math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS)):
        yield {k: np.sort(rng.choice(SRC[k]["tr"], h, replace=False)) for k in srcs}

class GRL(torch.autograd.Function):                      # gradient reversal for DANN
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None
def coral_loss(fa, fb):
    d = fa.size(1); return ((torch.cov(fa.T) - torch.cov(fb.T)) ** 2).sum() / (4 * d * d)

def train_dg(ts, meth, seed, lam_coral=1.0, lam_dann=0.1, eta_dro=0.01):     # defaults = your DenseNet runs (CORAL_W, DANN_W, DRO_ETA)
    assert meth != "ERM (natural)" and len(ts.split("+")) == 2
    srcs = ts.split("+"); b = mpath(ts, meth, seed); BEST, LAST, HIST, PRD = b + ".pt", b + "_last.pt", b + "_history.csv", b + "_preds.npz"
    if os.path.exists(PRD): print("skip (finished):", os.path.basename(b)); return
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model(); disc = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, 2)).to(dev) if meth == "DANN" else None
    opt = torch.optim.AdamW(list(m.parameters()) + (list(disc.parameters()) if disc else []), lr=LR, weight_decay=WD)
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); total, warm = EPOCHS * spe, min(300, spe)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda"); hist, best, start = [], 0.0, 0; q = torch.ones(2, device=dev) / 2
    if os.path.exists(LAST):
        ck = torch.load(LAST, map_location=dev, weights_only=False)
        m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); q = ck["q"].to(dev)
        if disc: disc.load_state_dict(ck["disc"])
        hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1; print("resumed, next epoch", start + 1)
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev); n = 0
        for step, batch in enumerate(tqdm(batches_bal(srcs, seed, ep), total=spe, leave=False, desc=f"ViT {ts} {meth} seed {seed} ep {ep + 1}/{EPOCHS}")):
            x = torch.cat([gpu_batch(SRC[k]["arr"], batch[k], flip=True) for k in srcs]); sizes = [len(batch[k]) for k in srcs]
            ys = [torch.from_numpy(SRC[k]["Y"][batch[k]]).to(dev).float() for k in srcs]
            mk = [torch.from_numpy(np.tile(SRC[k]["valid"], (len(batch[k]), 1))).to(dev).float() for k in srcs]
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda"):
                f = m.forward_head(m.forward_features(x), pre_logits=True); z = m.get_classifier()(f)
            z, f = z.float(), f.float(); zs, fs = torch.split(z, sizes), torch.split(f, sizes)
            gsum = [(F.binary_cross_entropy_with_logits(zz, yy, reduction="none") * mm).sum() for zz, yy, mm in zip(zs, ys, mk)]
            gden = [mm.sum() for mm in mk]
            if meth == "GroupDRO":                                   # exponentiated-gradient weights on the per-hospital losses
                gl = torch.stack([a / d for a, d in zip(gsum, gden)]); q = q * torch.exp(eta_dro * gl.detach()); q = q / q.sum(); loss = (q * gl).sum()
            else:
                loss = sum(gsum) / sum(gden)                         # masked pooled BCE (same as ERM)
                if meth == "CORAL": loss = loss + lam_coral * coral_loss(fs[0], fs[1])
                if meth == "DANN":                                   # 0.1 x domain loss with the lambda ramp, as in your DenseNet runs
                    ramp = 2 / (1 + math.exp(-10 * (ep * spe + step) / total)) - 1
                    dom = torch.cat([torch.full((s_,), i, device=dev) for i, s_ in enumerate(sizes)])
                    loss = loss + lam_dann * F.cross_entropy(disc(GRL.apply(f, ramp)), dom)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach() * len(x); n += len(x)
        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))     # same source-only, mean-over-hospitals selection for every method
        row = dict(epoch=ep + 1, train_loss=float(tl) / n, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"ViT {ts} {meth} seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f} | val {vs} | q={q.cpu().numpy().round(2) if meth == 'GroupDRO' else ''}")
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep, q=q.cpu(), **({"disc": disc.state_dict()} if disc else {})), LAST)
    m.load_state_dict(torch.load(BEST, map_location=dev, weights_only=True)); z = {}
    z["A_test_P"] = predict(m, NIH_A, te_i); z["B_test_P"] = predict(m, CX_T, np.arange(len(CX_T)))
    if "C" not in srcs: z["C_test_P"] = predict(m, VB_A, np.arange(NVB))
    for k in ["A", "B"] + (["C"] if "C" in srcs else []): z[f"{k}_val_P"] = predict(m, SRC[k]["arr"], SRC[k]["va"])
    np.savez(PRD, **z); del m, opt, sch, scaler; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import glob
stale = glob.glob(os.path.join(VT, "vit_ApB_seed0_dann*")); print("will delete:", [os.path.basename(f) for f in stale])
CONFIRM_DELETE = False                                  # set True once, after checking the list, then set back to False
if CONFIRM_DELETE:
    for f in stale: os.remove(f)

RUN_ORDER = ["A+C", "B+C", "A+B"]                       # A+C is fastest; A+B seed 0 is mostly done already
PLAN = [(ts, me, s) for ts in RUN_ORDER for me in METHS[1:] for s in SEEDS]
for ts in POOLED: assert all(os.path.exists(mpath(ts, "ERM (natural)", s) + "_preds.npz") for s in SEEDS), f"run Cell 3 first: ERM (natural) missing for {ts}"
done = lambda ts, me, s: os.path.exists(mpath(ts, me, s) + "_preds.npz")
display(pd.DataFrame({ts: {me: f"{sum(done(ts, me, s) for s in SEEDS)}/{len(SEEDS)}" for me in METHS} for ts in RUN_ORDER}))
STOP_AFTER = None                                       # e.g. 4 to run one batch at a time; rerun the cell to resume
for ts, me, s in [p for p in PLAN if not done(*p)][:STOP_AFTER]:
    try: train_dg(ts, me, s)
    except torch.cuda.OutOfMemoryError:
        print("OOM: set BS = 32 in Cell 0, restart, rerun Cells 0-2 and 13, then rerun this cell (it resumes)."); break
print("all finished" if not [p for p in PLAN if not done(*p)] else "runs left: rerun this cell")

In [ ]:
SETS_ORDER = ["A+B", "A+C", "B+C"]; SEEN_T = {"A+B": ["A", "B"], "A+C": ["A"], "B+C": ["B"]}
def load_pred(ts, me, s):
    f = mpath(ts, me, s) + "_preds.npz"; return dict(np.load(f)) if os.path.exists(f) else None
def thr_pred(z, ts):                                    # source-validation thresholds for this model
    thr = np.full(L, 0.5)
    for j in range(L):
        ks = [k for k in ts.split("+") if SRC[k]["valid"][j]]
        if ks: thr[j] = best_thr(np.concatenate([VAL_Y[k][:, j] for k in ks]), np.concatenate([z[f"{k}_val_P"][:, j] for k in ks]))
    return thr
rows, PDD = [], {}
for ts in POOLED:
    for me in METHS:
        for s in SEEDS:
            z = load_pred(ts, me, s)
            if z is None: continue
            PDD[(ts, me, s)] = z; T = UNSEEN[ts]; P, Y = z[f"{T}_test_P"], YT[T]; v = np.where(VALID[T])[0]; thr = thr_pred(z, ts)
            sa = [mauc(YT[t], z[f"{t}_test_P"], VALID[t]) for t in SEEN_T[ts]]
            rows.append(dict(train=ts, method=me, seed=s, unseen=T, auroc=mauc(Y, P, VALID[T]), auprc=mauprc(Y, P, VALID[T]),
                             ece=np.mean([ece_of(Y[:, j], P[:, j]) for j in v]), f1=np.mean([f1_at(Y[:, j], P[:, j], thr[j]) for j in v]),
                             seen_mean=np.mean(sa), seen_worst=np.min(sa)))
Q = pd.DataFrame(rows); Q.to_csv(os.path.join(VT, "table_vit_dg_per_seed.csv"), index=False)
MET = {"auroc": "Unseen AUROC", "auprc": "Unseen AUPRC", "ece": "Unseen ECE", "f1": "Unseen F1", "seen_mean": "Seen AUROC (mean)", "seen_worst": "Seen AUROC (worst)"}
g = Q.groupby(["train", "method"]); TAB = pd.DataFrame({MET[m]: g[m].agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}") for m in MET})
TAB["seeds"] = g.seed.nunique(); TAB = TAB.reindex(pd.MultiIndex.from_product([SETS_ORDER, METHS], names=["train", "method"])).dropna(how="all")
display(TAB); TAB.to_csv(os.path.join(VT, "table_vit_dg_main.csv"))
try: open(os.path.join(VT, "table_vit_dg_main.tex"), "w").write(TAB.to_latex(escape=True))
except Exception as e: print("LaTeX export skipped:", e)
print("\nGroupDRO final group weights q (0.5/0.5 = behaves like ERM balanced):")
for ts in POOLED:
    for s in SEEDS:
        f = mpath(ts, "GroupDRO", s) + "_last.pt"
        if os.path.exists(f): print(f"  {ts} seed {s}: q =", dict(zip(ts.split('+'), torch.load(f, map_location='cpu', weights_only=False)['q'].numpy().round(3))))

In [ ]:
rng = np.random.default_rng(5); NBQ = 500
SEEDS_OF = lambda ts, me: [s for s in SEEDS if (ts, me, s) in PDD]
PSm = {(ts, me): np.mean([PDD[(ts, me, s)][f"{UNSEEN[ts]}_test_P"] for s in SEEDS_OF(ts, me)], 0) for ts in POOLED for me in METHS if SEEDS_OF(ts, me)}
BQ = {k: np.zeros(NBQ) for k in PSm}
for r in tqdm(range(NBQ), desc="bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}                  # one resample per site, shared by all methods = paired
    for (ts, me), P in PSm.items(): T = UNSEEN[ts]; BQ[(ts, me)][r] = mauc(YT[T][ix[T]], P[ix[T]], VALID[T])
PT_ = {k: mauc(YT[UNSEEN[k[0]]], P, VALID[UNSEEN[k[0]]]) for k, P in PSm.items()}
rows = []
for ts in POOLED:
    if (ts, "ERM (natural)") not in PSm: continue
    for me in METHS[1:]:
        if (ts, me) not in PSm: continue
        d = BQ[(ts, me)] - BQ[(ts, "ERM (natural)")]
        rows.append(dict(train=ts, unseen=UNSEEN[ts], method=me, diff=PT_[(ts, me)] - PT_[(ts, "ERM (natural)")], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                         p=min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / NBQ)))
DS = pd.DataFrame(rows); o = np.argsort(DS.p.values); holm = np.empty(len(DS)); run = 0
for r_, i in enumerate(o): run = max(run, min(1, (len(DS) - r_) * DS.p.values[i])); holm[i] = run
DS["p_holm"] = holm; DS["significant"] = DS.p_holm < 0.05; display(DS.round(4)); DS.to_csv(os.path.join(VT, "table_vit_dg_vs_erm_natural.csv"), index=False)
print("Methods that beat ERM (natural) after Holm correction:", DS[(DS.significant) & (DS["diff"] > 0)][["train", "method"]].values.tolist() or "none")

In [ ]:
MC = {"ERM (natural)": "0.12", "ERM (balanced)": "0.38", "DANN": "0.58", "CORAL": "0.75", "GroupDRO": "0.9"}
HT = {"ERM (natural)": "", "ERM (balanced)": "///", "DANN": "\\\\\\", "CORAL": "xxx", "GroupDRO": "..."}
ci_ = lambda a: np.percentile(a, [2.5, 97.5]); w = 0.16

# Fig A: unseen-site AUROC (95% CI) and AUPRC (sd over seeds)
fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.0))
for i, me in enumerate(METHS):
    for k, ts in enumerate(SETS_ORDER):
        if (ts, me) not in PSm: continue
        x = k + (i - 2) * w; v = PT_[(ts, me)]; c = ci_(BQ[(ts, me)])
        ax[0].bar(x, v, w, yerr=[[max(v - c[0], 0)], [max(c[1] - v, 0)]], capsize=1.5, color=MC[me], hatch=HT[me], edgecolor="k", lw=.4, label=me if k == 0 else None)
        a = Q[(Q.train == ts) & (Q.method == me)].auprc; ax[1].bar(x, a.mean(), w, yerr=a.std(ddof=1) if len(a) > 1 else 0, capsize=1.5, color=MC[me], hatch=HT[me], edgecolor="k", lw=.4)
for a_, yl, t in zip(ax, ["Macro AUROC (unseen site)", "Macro AUPRC (unseen site)"], ["AUROC, 95% patient-bootstrap CI", "AUPRC, sd over seeds"]):
    a_.set_xticks(range(3)); a_.set_xticklabels([f"{t_}→{UNSEEN[t_]}\n({SITE[UNSEEN[t_]]})" for t_ in SETS_ORDER]); a_.set(ylabel=yl, title=t); a_.grid(True, axis="y", alpha=.3)
ax[0].set_ylim(max(.5, min(PT_.values()) - .05), None); ax[0].legend(frameon=False, fontsize=5.5, ncol=2); fig.tight_layout(); save_fig(fig, "fig_vit_dg_A_unseen_auroc_auprc")

# Fig B: forest plot, difference to ERM (natural)
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.6), sharex=True)
for a_, ts in zip(ax, SETS_ORDER):
    g_ = DS[DS.train == ts].reset_index(drop=True)
    for i, r in g_.iterrows():
        a_.errorbar(r["diff"], i, xerr=[[r["diff"] - r.lo], [r.hi - r["diff"]]], fmt="o", color="k", capsize=2, ms=4, mfc="k" if r.significant else "w")
        a_.text(r.hi + .0008, i, "*" if r.significant else "", va="center", fontsize=9)
    a_.axvline(0, color="k", lw=.6, ls="--"); a_.set_yticks(range(len(g_))); a_.set_yticklabels(g_.method if a_ is ax[0] else [""] * len(g_)); a_.invert_yaxis()
    a_.set(title=f"{ts}→{UNSEEN[ts]}", xlabel="AUROC − ERM (natural)"); a_.grid(True, axis="x", alpha=.3)
fig.suptitle("Difference to ERM (natural), 95% CI; filled marker/* = significant after Holm correction", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_vit_dg_B_forest_vs_erm")

# Fig C: seen vs unseen trade-off
MK = {"A+B": "o", "A+C": "s", "B+C": "^"}
fig, ax = plt.subplots(figsize=(COL1 * 1.5, 3.2))
for ts in SETS_ORDER:
    for me in METHS:
        d = Q[(Q.train == ts) & (Q.method == me)]
        if d.empty: continue
        ax.errorbar(d.seen_mean.mean(), d.auroc.mean(), xerr=d.seen_mean.std(ddof=1) if len(d) > 1 else 0, yerr=d.auroc.std(ddof=1) if len(d) > 1 else 0, fmt=MK[ts], color=MC[me], mec="k", mew=.5, ms=6, capsize=1.5, elinewidth=.6)
for me in METHS: ax.plot([], [], "s", color=MC[me], mec="k", label=me)
for ts in SETS_ORDER: ax.plot([], [], MK[ts], color="w", mec="k", label=f"{ts}→{UNSEEN[ts]}")
ax.set(xlabel="AUROC on seen hospitals (mean)", ylabel="AUROC on unseen hospital", title="In-distribution vs out-of-distribution"); ax.legend(frameon=False, fontsize=5.5, ncol=2); ax.grid(True, alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_dg_C_seen_vs_unseen")

# Fig D: per-label heatmaps on the unseen site
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.9), sharey=True)
for a_, ts in zip(axes, SETS_ORDER):
    ms_ = [me for me in METHS if (ts, me) in PSm]; T = UNSEEN[ts]
    D = np.array([lab_auc(YT[T], PSm[(ts, me)], VALID[T]) for me in ms_]); im = a_.imshow(D, cmap="viridis", vmin=.5, vmax=.95, aspect="auto")
    for i in range(D.shape[0]):
        for j in range(L): a_.text(j, i, "–" if np.isnan(D[i, j]) else f"{D[i, j]:.2f}", ha="center", va="center", color="w", fontsize=5.5)
    a_.set_xticks(range(L)); a_.set_xticklabels([l.replace("Pleural ", "Pl. ") for l in LABELS6], rotation=45, ha="right", fontsize=6); a_.set_title(f"{ts}→{UNSEEN[ts]}"); a_.grid(False)
    a_.set_yticks(range(len(ms_))); a_.set_yticklabels(ms_ if a_ is axes[0] else [""] * len(ms_))
fig.colorbar(im, ax=axes, label="AUROC", shrink=.8); save_fig(fig, "fig_vit_dg_D_per_label_heatmap")

In [ ]:
def reliab(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); xs_, ys_ = [], []
    for k in range(nb):
        mk = bi == k
        if mk.sum() > 30: xs_.append(p[mk].mean()); ys_.append(y[mk].mean())
    return xs_, ys_

# Fig E: ECE and F1 on the unseen site
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8))
for i, me in enumerate(METHS):
    for k, ts in enumerate(SETS_ORDER):
        d = Q[(Q.train == ts) & (Q.method == me)]
        if d.empty: continue
        for a_, m in zip(ax, ["ece", "f1"]):
            a_.bar(k + (i - 2) * w, d[m].mean(), w, yerr=d[m].std(ddof=1) if len(d) > 1 else 0, capsize=1.5, color=MC[me], hatch=HT[me], edgecolor="k", lw=.4, label=me if (k == 0 and m == "ece") else None)
for a_, t in zip(ax, ["ECE, unseen site (lower is better)", "Macro-F1 at source-tuned threshold (higher is better)"]):
    a_.set_xticks(range(3)); a_.set_xticklabels([f"{t_}→{UNSEEN[t_]}" for t_ in SETS_ORDER]); a_.set_title(t); a_.grid(True, axis="y", alpha=.3)
ax[0].legend(frameon=False, fontsize=5.5, ncol=2); fig.tight_layout(); save_fig(fig, "fig_vit_dg_E_ece_f1")

# Fig G: reliability curves on the unseen site
LSTY = {"ERM (natural)": "-", "ERM (balanced)": "--", "DANN": "-.", "CORAL": ":", "GroupDRO": (0, (5, 1, 1, 1))}
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.7), sharey=True)
for a_, ts in zip(axes, SETS_ORDER):
    T = UNSEEN[ts]; v = np.where(VALID[T])[0]
    for me in METHS:
        if (ts, me) not in PSm: continue
        xs_, ys_ = reliab(YT[T][:, v].ravel(), PSm[(ts, me)][:, v].ravel()); a_.plot(xs_, ys_, ls=LSTY[me], marker="o", ms=2.5, color=MC[me] if me != "GroupDRO" else "0.5", label=me)
    a_.plot([0, 1], [0, 1], "k:", lw=.6); a_.set(title=f"{ts}→{UNSEEN[ts]}", xlabel="Predicted probability"); a_.grid(True, alpha=.3)
axes[0].set_ylabel("Observed frequency"); axes[0].legend(frameon=False, fontsize=5.5); fig.tight_layout(); save_fig(fig, "fig_vit_dg_G_reliability")

# Fig F: per-source validation curves (shows overfitting to the small C source)
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.8))
for a_, ts in zip(axes, SETS_ORDER):
    srcs = ts.split("+")
    for me in METHS:
        f = mpath(ts, me, 0) + "_history.csv"
        if not os.path.exists(f): continue
        h = pd.read_csv(f)
        for k, src in enumerate(srcs): a_.plot(h.epoch, h[f"val_{src}"], ls="-" if k == 0 else "--", color=MC[me] if me != "GroupDRO" else "0.5", marker="o", ms=2, lw=.9)
    a_.set(title=f"{ts} (solid = {srcs[0]}, dashed = {srcs[1]})", xlabel="Epoch"); a_.grid(True, alpha=.3)
axes[0].set_ylabel("Source-validation AUROC"); [axes[0].plot([], [], color=MC[me] if me != "GroupDRO" else "0.5", label=me) for me in METHS]; axes[0].legend(frameon=False, fontsize=5.5)
fig.tight_layout(); save_fig(fig, "fig_vit_dg_F_training_curves_per_source")

import shutil; shutil.make_archive(os.path.join(OUT, "VIT_DG_bundle"), "zip", VT)
print("figures:", sorted(f for f in os.listdir(FIG) if f.startswith("fig_vit_dg") and f.endswith(".png")))

In [ ]:
P_ = {(ts, s): dict(np.load(base_path(ts, s) + "_preds.npz")) for ts in TS for s in SEEDS if os.path.exists(base_path(ts, s) + "_preds.npz")}
seeds_of = lambda ts: [s for s in SEEDS if (ts, s) in P_]
CELLS = [(ts, T) for ts in TS for T in "ABC" if not (T == "C" and "C" in ts.split("+"))]
PM = {(ts, T): np.mean([P_[(ts, s)][f"{T}_test_P"] for s in seeds_of(ts)], 0) for ts, T in CELLS}      # seed-averaged probabilities
seen = lambda ts, T: T in ts.split("+")

rows = []
for ts, T in CELLS:
    a = [mauc(YT[T], P_[(ts, s)][f"{T}_test_P"], VALID[T]) for s in seeds_of(ts)]
    p = [mauprc(YT[T], P_[(ts, s)][f"{T}_test_P"], VALID[T]) for s in seeds_of(ts)]
    rows.append(dict(train=ts, test=T, seen=seen(ts, T), auroc=np.mean(a), auroc_sd=np.std(a, ddof=1) if len(a) > 1 else 0, auprc=np.mean(p), auprc_sd=np.std(p, ddof=1) if len(p) > 1 else 0))
RES = pd.DataFrame(rows); RES.to_csv(os.path.join(VT, "table_vit_main.csv"), index=False)
fmt = lambda r, m: f"{r[m]:.3f} ± {r[m + '_sd']:.3f}" + (" †" if r.seen else "")
for m in ["auroc", "auprc"]:
    t = RES.assign(v=RES.apply(lambda r: fmt(r, m), axis=1)).pivot(index="train", columns="test", values="v").reindex(TS)
    print(f"\n=== ViT-S, {m.upper()} (mean ± sd over seeds; † = test site in training set; column C excludes Edema; NaN = C was a training source) ==="); display(t)

# ‡ reference for the C-containing rows: VinBigData validation split (also used for checkpoint selection, so optimistic)
rows = []
for ts in ["C", "A+C", "B+C"]:
    v = [mauc(Yvb[vb_va], P_[(ts, s)]["C_val_P"], VALID_C) for s in seeds_of(ts)]
    rows.append(dict(train=ts, test="C (val split ‡)", auroc=f"{np.mean(v):.3f} ± {np.std(v, ddof=1) if len(v) > 1 else 0:.3f}"))
display(pd.DataFrame(rows))

In [ ]:
NB = 300; rng = np.random.default_rng(0)
BSL = {c: np.zeros((NB, L)) for c in CELLS}
for r in tqdm(range(NB), desc="bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}                 # one resample per site, shared by all models = paired
    for ts, T in CELLS: BSL[(ts, T)][r] = lab_auc(YT[T][ix[T]], PM[(ts, T)][ix[T]], VALID[T])
BSM = {c: np.nanmean(v, 1) for c, v in BSL.items()}
PTL = {c: lab_auc(YT[c[1]], PM[c], VALID[c[1]]) for c in CELLS}; PTM = {c: np.nanmean(v) for c, v in PTL.items()}
ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)

rows = [dict(train=ts, test=T, auroc=PTM[(ts, T)], lo=ci(BSM[(ts, T)])[0], hi=ci(BSM[(ts, T)])[1], seen=seen(ts, T)) for ts, T in CELLS]
CI = pd.DataFrame(rows); CI.to_csv(os.path.join(VT, "table_vit_auroc_ci.csv"), index=False); display(CI.round(3))

gap = {"A→B (NIH→CheXpert)": ("A", "B"), "B→A (CheXpert→NIH)": ("B", "A")}      # gap = in-hospital AUROC − external AUROC, 6 labels
GB, GP = {}, {}
for nm, (s_, t_) in gap.items(): GB[nm] = BSL[(s_, s_)] - BSL[(s_, t_)]; GP[nm] = PTL[(s_, s_)] - PTL[(s_, t_)]
k1, k2 = list(gap)
asym_b = np.nanmean(GB[k1], 1) - np.nanmean(GB[k2], 1); asym_p = np.nanmean(GP[k1]) - np.nanmean(GP[k2])
rows = [dict(quantity=k, mean_gap=np.nanmean(GP[k]), lo=np.percentile(np.nanmean(GB[k], 1), 2.5), hi=np.percentile(np.nanmean(GB[k], 1), 97.5)) for k in gap]
rows.append(dict(quantity="Asymmetry (A→B gap − B→A gap)", mean_gap=asym_p, lo=np.percentile(asym_b, 2.5), hi=np.percentile(asym_b, 97.5)))
GAPT = pd.DataFrame(rows); display(GAPT.round(3)); GAPT.to_csv(os.path.join(VT, "table_vit_gaps.csv"), index=False)
print("Asymmetry CI excludes 0 ->", not (GAPT.lo.iloc[-1] <= 0 <= GAPT.hi.iloc[-1]))

In [ ]:
# Fig 1: training curves of the ERM (natural) runs
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
for ts in TS:
    for s in seeds_of(ts):
        h = pd.read_csv(base_path(ts, s) + "_history.csv"); c = f"C{TS.index(ts)}"
        ax[0].plot(h.epoch, h.train_loss, color=c, alpha=.6, label=ts if s == seeds_of(ts)[0] else None); ax[1].plot(h.epoch, h.val_mean, color=c, alpha=.6)
ax[0].set(xlabel="Epoch", ylabel="Train loss", title="Training loss"); ax[1].set(xlabel="Epoch", ylabel="Source-val AUROC", title="Validation AUROC (source only)")
ax[0].legend(frameon=False, ncol=2); [a.grid(True, alpha=.3) for a in ax]; fig.tight_layout(); save_fig(fig, "fig_vit_training_curves")

# Fig 2: transfer matrix (rows = training set, columns = test site)
M = np.array([[PTM.get((ts, T), np.nan) for T in "ABC"] for ts in TS])
fig, ax = plt.subplots(figsize=(COL1 * 1.3, 3.0)); im = ax.imshow(M, cmap="Greys", vmin=np.nanmin(M) - .03, vmax=np.nanmax(M) + .02)
for i, ts in enumerate(TS):
    for j, T in enumerate("ABC"):
        if np.isnan(M[i, j]): ax.text(j, i, "–", ha="center", va="center"); continue
        ax.text(j, i, f"{M[i, j]:.3f}" + ("†" if seen(ts, T) else ""), ha="center", va="center", color="white" if M[i, j] > np.nanmean(M) else "black")
ax.set_xticks(range(3)); ax.set_xticklabels([f"{SITE[t]}\n({t})" + ("\n5 findings" if t == "C" else "") for t in "ABC"]); ax.set_yticks(range(len(TS))); ax.set_yticklabels(TS)
ax.set(xlabel="Tested on", ylabel="Trained on", title="ViT-S AUROC († = site seen in training)"); ax.grid(False); fig.tight_layout(); save_fig(fig, "fig_vit_transfer_matrix")

# Fig 3: AUROC with 95% CI for every external direction
ext = [(ts, T) for ts, T in CELLS if not seen(ts, T)]; xs = np.arange(len(ext))
fig, ax = plt.subplots(figsize=(COL2, 2.9)); v = np.array([PTM[c] for c in ext]); c_ = np.array([ci(BSM[c]) for c in ext])
ax.bar(xs, v, 0.65, yerr=[np.clip(v - c_[:, 0], 0, None), np.clip(c_[:, 1] - v, 0, None)], capsize=2, color=["0.3" if "+" not in a else "0.7" for a, _ in ext], edgecolor="k", lw=.4)
ax.set_xticks(xs); ax.set_xticklabels([f"{a}→{b}" for a, b in ext], rotation=40, ha="right"); ax.set_ylim(max(0.5, v.min() - .08), v.max() + .04)
ax.set(ylabel="Macro AUROC (external site)", title="ViT-S external performance (dark = single source, light = pooled sources)"); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_external_auroc")

# Fig 4: per-label AUROC heatmap, external directions
D = np.array([PTL[c] for c in ext]); fig, ax = plt.subplots(figsize=(COL2, 3.2)); im = ax.imshow(D, cmap="viridis", aspect="auto")
for i in range(D.shape[0]):
    for j in range(L): ax.text(j, i, "–" if np.isnan(D[i, j]) else f"{D[i, j]:.2f}", ha="center", va="center", color="w", fontsize=6)
ax.set_xticks(range(L)); ax.set_xticklabels(LABELS6, rotation=30, ha="right"); ax.set_yticks(range(len(ext))); ax.set_yticklabels([f"{a}→{b}" for a, b in ext]); ax.grid(False)
fig.colorbar(im, ax=ax, label="AUROC"); ax.set_title("Per-disease external AUROC (ViT-S)"); fig.tight_layout(); save_fig(fig, "fig_vit_per_label_heatmap")

# Fig 5: gaps, asymmetry, per-label gaps
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8), gridspec_kw=dict(width_ratios=[1.6, 1])); x = np.arange(L)
for o, (nm, c) in zip([-.2, .2], zip(gap, ["0.35", "0.75"])):
    pt = GP[nm]; cc = ci(GB[nm]); ax[0].bar(x + o, pt, .4, yerr=[np.clip(pt - cc[0], 0, None), np.clip(cc[1] - pt, 0, None)], capsize=1.5, color=c, edgecolor="k", lw=.4, label=nm)
ax[0].axhline(0, color="k", lw=.6); ax[0].set_xticks(x); ax[0].set_xticklabels(LABELS6, rotation=35, ha="right"); ax[0].set(ylabel="AUROC drop (in-hospital − external)"); ax[0].legend(frameon=False); ax[0].grid(True, axis="y", alpha=.3)
g = GAPT; ax[1].bar(range(3), g.mean_gap, .6, yerr=[np.clip(g.mean_gap - g.lo, 0, None), np.clip(g.hi - g.mean_gap, 0, None)], capsize=2, color=["0.35", "0.75", "0.55"], edgecolor="k", lw=.4)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(["A→B", "B→A", "Asym."]); ax[1].axhline(0, color="k", lw=.6); ax[1].set(ylabel="Mean AUROC difference"); ax[1].grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_gaps_asymmetry")

In [ ]:
def thr_for(ts, s):
    thr = np.full(L, 0.5)
    for j in range(L):
        ks = [k for k in ts.split("+") if SRC[k]["valid"][j]]
        if ks: thr[j] = best_thr(np.concatenate([VAL_Y[k][:, j] for k in ks]), np.concatenate([P_[(ts, s)][f"{k}_val_P"][:, j] for k in ks]))
    return thr
rows = []
for ts, T in CELLS:
    for s in seeds_of(ts):
        P = P_[(ts, s)][f"{T}_test_P"]; Y = YT[T]; v = np.where(VALID[T])[0]; thr = thr_for(ts, s)      # source-tuned thresholds, applied unchanged to the target
        rows.append(dict(train=ts, test=T, seed=s, seen=seen(ts, T), ece=np.mean([ece_of(Y[:, j], P[:, j]) for j in v]), f1=np.mean([f1_at(Y[:, j], P[:, j], thr[j]) for j in v])))
CAL = pd.DataFrame(rows); CAL.to_csv(os.path.join(VT, "table_vit_calibration.csv"), index=False)
for m in ["ece", "f1"]:
    t = CAL.groupby(["train", "test"])[m].agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}").unstack("test").reindex(TS)
    print(f"\n=== ViT-S {m.upper()} (source-tuned thresholds) ==="); display(t)

cases = [("A", "A", "A→A (in-hospital)"), ("A", "B", "A→B"), ("B", "B", "B→B (in-hospital)"), ("B", "A", "B→A"), ("A+B", "C", "A+B→C"), ("A+C", "B", "A+C→B")]
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.6), sharey=True)
for a_, grp, ttl in zip(ax, [cases[:2], cases[2:4], cases[4:]], ["Trained on NIH", "Trained on CheXpert", "Pooled models"]):
    for ts, T, nm in grp:
        v = np.where(VALID[T])[0]; xs_, ys_ = reliab(YT[T][:, v].ravel(), P_[(ts, 0)][f"{T}_test_P"][:, v].ravel()); a_.plot(xs_, ys_, "-o", ms=3, label=nm)
    a_.plot([0, 1], [0, 1], "k:", lw=.6); a_.set(title=ttl, xlabel="Predicted probability"); a_.legend(frameon=False); a_.grid(True, alpha=.3)
ax[0].set_ylabel("Observed frequency"); fig.tight_layout(); save_fig(fig, "fig_vit_reliability")

t = CAL[~CAL.seen].groupby(["train", "test"])[["ece", "f1"]].mean().reset_index(); lab = [f"{a}→{b}" for a, b in zip(t.train, t.test)]
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
for a_, m, ttl in zip(ax, ["ece", "f1"], ["ECE (lower is better)", "Macro-F1, source threshold (higher is better)"]):
    a_.bar(range(len(t)), t[m], color="0.55", edgecolor="k", lw=.4); a_.set_xticks(range(len(t))); a_.set_xticklabels(lab, rotation=40, ha="right"); a_.set_title(ttl); a_.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_ece_f1_external")

In [ ]:
def fit_platt(z, y):
    lr = LogisticRegression(C=1e6, max_iter=500).fit(z.reshape(-1, 1), y); return float(np.clip(lr.coef_[0, 0], .1, 5)), float(lr.intercept_[0])
def correct(meth, zp, yp, ze, t0):
    a, b, t = 1.0, 0.0, t0; npos = int(yp.sum()); nneg = len(yp) - npos
    if npos >= 5 and nneg >= 5:
        if meth == "Platt + threshold": a, b = fit_platt(zp, yp)
        t = best_thr(yp, sig(a * zp + b))
    return sig(a * ze + b), t
KS, DRAWS, METH = [0, 25, 50, 100, 250, 500, 1000, 2500, 5000], 20, ["Threshold only", "Platt + threshold"]
POOL = {"A": ("B", VAL_Y["B"], "B_val_P"), "B": ("A", VAL_Y["A"], "A_val_P")}        # model trained on A -> pool = CheXpert val; B -> NIH val
rows, ref = [], []
for ts in ["A", "B"]:
    T, Yp, key = POOL[ts]
    for s in seeds_of(ts):
        z = P_[(ts, s)]; Pe, Ye, Pp = z[f"{T}_test_P"], YT[T], z[key]; Ze, Zp = logit(Pe), logit(Pp); thr0 = thr_for(ts, s)
        Pi, Yi = z[f"{ts}_test_P"], YT[ts]
        ref.append(dict(ts=ts, seed=s, ece_in=np.mean([ece_of(Yi[:, j], Pi[:, j]) for j in range(L)]), f1_in=np.mean([f1_at(Yi[:, j], Pi[:, j], thr0[j]) for j in range(L)])))
        for K in KS:
            for dr in range(1 if K == 0 else DRAWS):
                idx = np.random.default_rng([s, K, dr]).choice(len(Yp), K, replace=False) if K else np.array([], int)
                for me in METH:
                    es, fs = [], []
                    for j in range(L):
                        pe, t = correct(me, Zp[idx, j], Yp[idx, j], Ze[:, j], thr0[j]) if K else (sig(Ze[:, j]), thr0[j])
                        es.append(ece_of(Ye[:, j], pe)); fs.append(f1_at(Ye[:, j], pe, t))
                    rows.append(dict(ts=ts, seed=s, method=me, K=K, draw=dr, ece=np.mean(es), f1=np.mean(fs)))
RC, RF = pd.DataFrame(rows), pd.DataFrame(ref); RC.to_csv(os.path.join(VT, "recal_raw.csv"), index=False)
ps = RC.groupby(["ts", "method", "K", "seed"])[["ece", "f1"]].mean().reset_index(); refm = RF.groupby("ts")[["ece_in", "f1_in"]].mean()
tab = ps[ps.K.isin([0, 100, 500, 2500])].groupby(["ts", "method", "K"])[["ece", "f1"]].mean().round(3).unstack("K"); display(tab); tab.to_csv(os.path.join(VT, "table_vit_recal.csv"))

fig, ax = plt.subplots(2, 2, figsize=(COL2, 5.0)); DN = {"A": "A→B (NIH→CheXpert)", "B": "B→A (CheXpert→NIH)"}
for r_, ts in enumerate(["A", "B"]):
    for c_, (m, ttl) in enumerate([("ece", "ECE (lower is better)"), ("f1", "Macro-F1 (higher is better)")]):
        for me, col in zip(METH, ["C0", "C3"]):
            g = ps[(ps.ts == ts) & (ps.method == me)].groupby("K")[m].agg(["mean", "std"])
            ax[r_, c_].plot(g.index, g["mean"], "-o", ms=3, color=col, label=me); ax[r_, c_].fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"], color=col, alpha=.15, lw=0)
        ax[r_, c_].axhline(refm.loc[ts, m + "_in"], color="k", ls="--", lw=.7, label="in-hospital reference"); ax[r_, c_].set_xscale("symlog", linthresh=25)
        ax[r_, c_].set(xlabel="k labelled target images", title=f"{DN[ts]}: {ttl}"); ax[r_, c_].grid(True, alpha=.3)
ax[0, 0].legend(frameon=False); fig.tight_layout(); save_fig(fig, "fig_vit_recalibration")

In [ ]:
# ---- E1: seed ensemble vs single seed
rows = []
for ts, T in CELLS:
    if len(seeds_of(ts)) < 2: continue
    sg = [mauc(YT[T], P_[(ts, s)][f"{T}_test_P"], VALID[T]) for s in seeds_of(ts)]
    rows.append(dict(direction=f"{ts}→{T}", single_mean=np.mean(sg), seed_ensemble=PTM[(ts, T)], gain=PTM[(ts, T)] - np.mean(sg)))
E1 = pd.DataFrame(rows); display(E1.round(4)); E1.to_csv(os.path.join(VT, "table_ens_seed.csv"), index=False)

# ---- E2: cross-source ensemble vs pooled training (equal member count)
TGT = {"A": (["B", "C"], "B+C"), "B": (["A", "C"], "A+C"), "C": (["A", "B"], "A+B")}
S3 = [s for s in SEEDS if all((ts, s) in P_ for ts in TS)]; assert len(S3) >= 2, "need >= 2 complete seeds"
def members(T):
    srcs, pool = TGT[T]
    cross = [np.mean([P_[(k, s)][f"{T}_test_P"] for k in srcs], 0) for s in S3]                                  # 2 single-source models per seed
    pool1 = [P_[(pool, s)][f"{T}_test_P"] for s in S3]                                                           # 1 pooled model per seed
    pool2 = [np.mean([P_[(pool, s)][f"{T}_test_P"], P_[(pool, S3[(i + 1) % len(S3)])][f"{T}_test_P"]], 0) for i, s in enumerate(S3)]   # 2 pooled models = same inference cost
    return dict(cross=cross, pool1=pool1, pool2=pool2)
MB = {T: members(T) for T in "ABC"}; rng = np.random.default_rng(1); NB2 = 300
ens_auc = lambda T, ix, k: np.mean([mauc(YT[T][ix], p[ix], VALID[T]) for p in MB[T][k]])
pt = {(T, k): ens_auc(T, np.arange(len(YT[T])), k) for T in "ABC" for k in ["cross", "pool1", "pool2"]}
bs = {(T, k): np.zeros(NB2) for T in "ABC" for k in ["cross", "pool1", "pool2"]}
for r in tqdm(range(NB2), desc="ensemble bootstrap"):
    for T in "ABC":
        ix = resample(T, rng)
        for k in ["cross", "pool1", "pool2"]: bs[(T, k)][r] = ens_auc(T, ix, k)
rows = []
for T in "ABC":
    for ref_, nm in [("pool1", "pooled ×1"), ("pool2", "pooled ×2 (equal cost)")]:
        d = bs[(T, "cross")] - bs[(T, ref_)]; p = min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / NB2)
        rows.append(dict(target=f"{SITE[T]} ({T})", comparison=f"cross-source ensemble − {nm}", diff=pt[(T, "cross")] - pt[(T, ref_)], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=p))
E2 = pd.DataFrame(rows); o = np.argsort(E2.p.values); holm = np.empty(len(E2)); run = 0
for r_, i in enumerate(o): run = max(run, min(1, (len(E2) - r_) * E2.p.values[i])); holm[i] = run
E2["p_holm"] = holm; display(E2.round(4)); E2.to_csv(os.path.join(VT, "table_ens_cross_source.csv"), index=False)
fig, ax = plt.subplots(figsize=(COL2, 2.9)); w = .26
for i, (k, nm, c) in enumerate([("pool1", "Pooled ×1", "0.75"), ("pool2", "Pooled ×2", "0.5"), ("cross", "Cross-source ensemble", "0.2")]):
    v = np.array([pt[(T, k)] for T in "ABC"]); cc = np.array([np.percentile(bs[(T, k)], [2.5, 97.5]) for T in "ABC"])
    ax.bar(np.arange(3) + (i - 1) * w, v, w, yerr=[np.clip(v - cc[:, 0], 0, None), np.clip(cc[:, 1] - v, 0, None)], capsize=2, color=c, edgecolor="k", lw=.4, label=nm)
ax.set_xticks(range(3)); ax.set_xticklabels([f"Test {SITE[T]}\n(sources: {'+'.join(TGT[T][0])})" for T in "ABC"]); ax.set_ylim(.6, None); ax.set(ylabel="Macro AUROC"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_ensemble_cross_source")

# ---- E3 (optional): ViT + ConvNeXt on targets A and B
XD = os.path.join(OUT, "part34"); E_OLD = 7
def cnx(src, s): return np.load(os.path.join(XD, f"convnext_tiny_{src}_e{E_OLD}_seed{s}_preds.npz"))
try:
    CNX = {"B": np.mean([cnx("nih", s)["cx_test_P"] for s in SEEDS], 0), "A": np.mean([cnx("cx", s)["nih_test_P"] for s in SEEDS], 0)}
    VIT = {"B": PM[("A", "B")], "A": PM[("B", "A")]}; rows = []; rng = np.random.default_rng(2)
    for T, nm in [("B", "A→B"), ("A", "B→A")]:
        mods = {"ViT-S": VIT[T], "ConvNeXt-T": CNX[T], "ViT-S + ConvNeXt-T": (VIT[T] + CNX[T]) / 2}; bsE = {k: [] for k in mods}
        for _ in range(NB2):
            ix = resample(T, rng)
            for k, p in mods.items(): bsE[k].append(mauc(YT[T][ix], p[ix], VALID[T]))
        for k, p in mods.items(): rows.append(dict(direction=nm, model=k, auroc=mauc(YT[T], p, VALID[T]), lo=np.percentile(bsE[k], 2.5), hi=np.percentile(bsE[k], 97.5)))
    E3 = pd.DataFrame(rows); display(E3.round(3)); E3.to_csv(os.path.join(VT, "table_ens_architecture.csv"), index=False)
    fig, ax = plt.subplots(figsize=(COL1 * 1.5, 2.7)); w = .26
    for i, (k, c) in enumerate(zip(["ViT-S", "ConvNeXt-T", "ViT-S + ConvNeXt-T"], ["0.8", "0.5", "0.2"])):
        g = E3[E3.model == k]; ax.bar(np.arange(2) + (i - 1) * w, g.auroc, w, yerr=[g.auroc - g.lo, g.hi - g.auroc], capsize=2, color=c, edgecolor="k", lw=.4, label=k)
    ax.set_xticks(range(2)); ax.set_xticklabels(["A→B", "B→A"]); ax.set_ylim(.6, None); ax.set(ylabel="External macro AUROC"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
    fig.tight_layout(); save_fig(fig, "fig_vit_ensemble_architecture")
except FileNotFoundError as e: print("E3 skipped (ConvNeXt files not found):", e)

In [ ]:
class ViTCAM:
    def __init__(self, m):
        self.m = m; self.np_ = m.num_prefix_tokens; m.blocks[-1].norm1.register_forward_hook(self._fwd)
    def _fwd(self, mod, i, out):
        self.act = out
        if out.requires_grad: out.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, x, j):                                   # j: int or LongTensor [B] (label per image)
        self.m.zero_grad(set_to_none=True)
        with torch.enable_grad():
            z = self.m(x); j = torch.full((len(x),), j, device=x.device) if isinstance(j, int) else j
            z.gather(1, j.view(-1, 1)).sum().backward()
        a, g = self.act[:, self.np_:].detach(), self.grad[:, self.np_:].detach()
        cam = torch.relu((g.mean(1, keepdim=True) * a).sum(-1)); n = int(cam.shape[1] ** .5)
        return F.interpolate(cam.view(-1, 1, n, n), size=224, mode="bilinear", align_corners=False)[:, 0].cpu().numpy(), torch.sigmoid(z.detach()).cpu().numpy()
def load_vit(ts, s):
    m = make_model(False); m.load_state_dict(torch.load(base_path(ts, s) + ".pt", map_location=dev, weights_only=True)); return m.eval()
def cam_batch(cam, A, idx, j): return cam(gpu_batch(A, idx), j)
def norm_cam(c): return c / (c.max() + 1e-8)

CASES = [("A", "B", "A→B (NIH model on CheXpert)"), ("B", "A", "B→A (CheXpert model on NIH)"), ("A+B", "C", "A+B→C (pooled model on VinBigData)")]
for ts, T, ttl in CASES:                                         # TP / FP / FN examples, 2 diseases per direction
    m = load_vit(ts, 0); cam = ViTCAM(m); A_, idxs = TEST[T]; P = P_[(ts, 0)][f"{T}_test_P"]; Y = YT[T]
    for lab in ["Cardiomegaly", "Pleural Effusion"]:
        j = LABELS6.index(lab); p, y = P[:, j], Y[:, j]
        pick = {"True positive": np.where(y == 1)[0][np.argsort(-p[y == 1])[:4]], "False positive": np.where(y == 0)[0][np.argsort(-p[y == 0])[:4]], "False negative": np.where(y == 1)[0][np.argsort(p[y == 1])[:4]]}
        fig, ax = plt.subplots(3, 4, figsize=(COL2, 5.4))
        for r_, (nm, ii) in enumerate(pick.items()):
            cams, probs = cam_batch(cam, A_, idxs[ii], j)
            for c_ in range(4):
                a_ = ax[r_, c_]; a_.imshow(A_[idxs[ii[c_]]], cmap="gray"); a_.imshow(norm_cam(cams[c_]), cmap="jet", alpha=.4); a_.axis("off"); a_.set_title(f"{nm}\np={probs[c_, j]:.2f}", fontsize=6)
        fig.suptitle(f"{ttl} · {lab}", fontsize=8); fig.tight_layout(); save_fig(fig, f"fig_vit_gradcam_{ts.replace('+', 'p')}_to_{T}_{lab.replace(' ', '')}")
    del m, cam; torch.cuda.empty_cache()

def border_share(c, f=.1):
    k = int(224 * f); m_ = np.ones((224, 224), bool); m_[k:-k, k:-k] = False; return float(c[m_].sum() / (c.sum() + 1e-8))
rng = np.random.default_rng(0); rows = []
for ts, T, ttl in CASES:
    m = load_vit(ts, 0); cam = ViTCAM(m); A_, idxs = TEST[T]; P = P_[(ts, 0)][f"{T}_test_P"]; sel = np.sort(rng.choice(len(idxs), 300, replace=False))
    for i in range(0, len(sel), 32):
        ii = sel[i:i + 32]; j = torch.from_numpy(np.where(VALID[T][None], P[ii], -1).argmax(1)).to(dev)         # CAM of each image's top-scored finding
        cams, _ = cam_batch(cam, A_, idxs[ii], j); rows += [dict(case=ttl.split(" (")[0], border=border_share(c)) for c in cams]
    del m, cam; torch.cuda.empty_cache()
BR = pd.DataFrame(rows); print("uniform CAM would put 36% of its mass in the outer 10% border"); display(BR.groupby("case").border.agg(["mean", "std"]).round(3)); BR.to_csv(os.path.join(VT, "table_gradcam_border.csv"), index=False)
fig, ax = plt.subplots(figsize=(COL1 * 1.4, 2.6)); cs = list(BR.case.unique()); ax.boxplot([BR[BR.case == c].border for c in cs], labels=cs, showfliers=False)
ax.axhline(.36, color="r", ls="--", lw=.7, label="uniform CAM"); ax.set(ylabel="CAM mass in outer 10% border"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_vit_gradcam_border")

In [ ]:
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score
m = load_vit("A+B", 0); rng = np.random.default_rng(0); feats, site = [], []
@torch.inference_mode()
def embed(A, idx):
    out = []
    for i in range(0, len(idx), 128):
        with torch.autocast("cuda"): out.append(m.forward_head(m.forward_features(gpu_batch(A, idx[i:i + 128])), pre_logits=True).float().cpu())
    return torch.cat(out).numpy()
for T in "ABC":
    A_, idxs = TEST[T]; ii = np.sort(rng.choice(len(idxs), 600, replace=False)); feats.append(embed(A_, idxs[ii])); site += [T] * 600
X = np.concatenate(feats); site = np.array(site); Z2 = TSNE(2, perplexity=35, random_state=0, init="pca").fit_transform(X)
acc = cross_val_score(LogisticRegression(max_iter=2000), X, site, cv=5).mean()
fig, ax = plt.subplots(figsize=(COL1 * 1.5, 3.2))
for T, c in zip("ABC", ["C0", "C1", "C2"]): ax.scatter(*Z2[site == T].T, s=4, alpha=.6, color=c, label=f"{SITE[T]} ({T})")
ax.set(title=f"ViT-S (A+B) embeddings by hospital · site-probe accuracy {acc:.2f} (chance 0.33)", xticks=[], yticks=[]); ax.legend(frameon=False, markerscale=3)
fig.tight_layout(); save_fig(fig, "fig_vit_tsne_site"); print("site-probe accuracy:", round(acc, 3)); del m; torch.cuda.empty_cache()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
FE = os.path.join(OUT, "fair_ens"); CNXDG = os.path.join(OUT, "convnext_dg"); NAT = "ERM (natural)"
FN = {"DenseNet121": "DenseNet", "ConvNeXt-T": "ConvNeXt"}
TAGF = {"ERM (natural)": "erm_nat", "ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral"}       # GroupDRO is not in fair_ens
TAGC = {"ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral", "GroupDRO": "gdro"}               # ConvNeXt DG runs from the separate notebook
A3 = ["DenseNet121", "ConvNeXt-T", "ViT-S"]; AC = {"DenseNet121": "0.2", "ConvNeXt-T": "0.55", "ViT-S": "0.85"}; ALS = {"DenseNet121": "-", "ConvNeXt-T": "--", "ViT-S": ":"}
NAMEM = {"DenseNet121": "densenet121", "ConvNeXt-T": "convnext_tiny.fb_in1k", "ViT-S": ARCH}

def probs3(a, ts, me, s):
    if a == "ViT-S":
        z = load_pred(ts, me, s); return None if z is None else {T: z[f"{T}_test_P"] for T in "ABC" if f"{T}_test_P" in z}
    if a == "ConvNeXt-T" and me != NAT:
        if me not in TAGC: return None
        f = os.path.join(CNXDG, f"cnx_{ts.replace('+', 'p')}_seed{s}_{TAGC[me]}_preds.npz")
        if not os.path.exists(f): return None
        z = np.load(f); return {T: z[f"{T}_test_P"] for T in "ABC" if f"{T}_test_P" in z}
    tag = TAGF.get(me)
    if tag is None: return None                                                  # e.g. DenseNet GroupDRO: no VinBigData predictions
    f = os.path.join(FE, f"logits_{FN[a]}_{ts.replace('+', 'p')}_{tag}_seed{s}.npz")
    if not os.path.exists(f): return None
    z = np.load(f); return {T: sig(z[T]) for T in "ABC" if not (T == "C" and "C" in ts.split("+"))}

PR3 = {}
for a in A3:
    for ts in TS:
        for me in METHS:
            d = {s: probs3(a, ts, me, s) for s in SEEDS}; d = {s: v for s, v in d.items() if v is not None}
            if d: PR3[(a, ts, me)] = d
cov = pd.DataFrame([dict(arch=a, train=ts, method=me, seeds=len(d)) for (a, ts, me), d in PR3.items()]).pivot_table(index=["train", "method"], columns="arch", values="seeds", fill_value=0)
print("seeds available per model (0 = not trained):"); display(cov)

EXT = [(ts, T) for ts in TS for T in "ABC" if T not in ts.split("+")]       # 9 external directions
SEENC = [(ts, T) for ts in TS for T in "AB" if T in ts.split("+")]          # in-distribution cells that have a clean held-out test set
ALLC = EXT + SEENC
def pm(a, ts, T, me=NAT): d = PR3[(a, ts, me)]; return np.mean([d[s][T] for s in d], 0)        # seed-averaged probabilities
def dname(ts, T): return f"{ts}→{T}"

In [ ]:
rows = []
for a in A3:
    for ts, T in ALLC:
        if (a, ts, NAT) not in PR3: continue
        d = PR3[(a, ts, NAT)]
        v = [mauc(YT[T], d[s][T], VALID[T]) for s in d]; p = [mauprc(YT[T], d[s][T], VALID[T]) for s in d]
        rows.append(dict(arch=a, train=ts, test=T, seen=T in ts.split("+"),
                         auroc=np.mean(v), auroc_sd=np.std(v, ddof=1) if len(v) > 1 else 0,
                         auprc=np.mean(p), auprc_sd=np.std(p, ddof=1) if len(p) > 1 else 0))
R3 = pd.DataFrame(rows); R3.to_csv(os.path.join(VT, "table_arch3_main.csv"), index=False)
for m in ["auroc", "auprc"]:
    t = R3.assign(v=R3.apply(lambda r: f"{r[m]:.3f} ± {r[m + '_sd']:.3f}" + (" †" if r.seen else ""), axis=1)).pivot(index="train", columns=["test", "arch"], values="v").reindex(TS)
    print(f"\n=== {m.upper()} by architecture (ERM natural, mean ± sd over seeds; † = seen site; column C = 5 findings) ==="); display(t)

M = {a: np.array([[R3[(R3.arch == a) & (R3.train == ts) & (R3.test == T)].auroc.mean() if len(R3[(R3.arch == a) & (R3.train == ts) & (R3.test == T)]) else np.nan for T in "ABC"] for ts in TS]) for a in A3}
fig, axes = plt.subplots(2, 3, figsize=(COL2, 5.6)); lo_, hi_ = np.nanmin(list(M.values())) - .02, np.nanmax(list(M.values())) + .01
for a_, a in zip(axes[0], A3):
    a_.imshow(M[a], cmap="Greys", vmin=lo_, vmax=hi_)
    for i, ts in enumerate(TS):
        for j, T in enumerate("ABC"):
            if not np.isnan(M[a][i, j]): a_.text(j, i, f"{M[a][i, j]:.3f}" + ("†" if T in ts.split("+") else ""), ha="center", va="center", fontsize=6, color="w" if M[a][i, j] > (lo_ + hi_) / 2 else "k")
    a_.set_title(a); a_.set_xticks(range(3)); a_.set_xticklabels(["A", "B", "C*"]); a_.set_yticks(range(len(TS))); a_.set_yticklabels(TS if a_ is axes[0, 0] else [""] * len(TS)); a_.grid(False)
for a_, (b, a) in zip(axes[1], [("ConvNeXt-T", "DenseNet121"), ("ViT-S", "DenseNet121"), ("ViT-S", "ConvNeXt-T")]):
    D = M[b] - M[a]; mx = np.nanmax(np.abs(D)) + 1e-6; im2 = a_.imshow(D, cmap="RdBu", vmin=-mx, vmax=mx)
    for i in range(len(TS)):
        for j in range(3):
            if not np.isnan(D[i, j]): a_.text(j, i, f"{D[i, j]:+.3f}", ha="center", va="center", fontsize=6)
    a_.set_title(f"{b} − {a}"); a_.set_xticks(range(3)); a_.set_xticklabels(["A", "B", "C*"]); a_.set_yticks(range(len(TS))); a_.set_yticklabels(TS if a_ is axes[1, 0] else [""] * len(TS)); a_.grid(False)
    fig.colorbar(im2, ax=a_, shrink=.7)
fig.suptitle("Top: AUROC (rows = training set, columns = test site; † seen, *5 findings). Bottom: architecture differences", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_arch3_transfer_matrices")

In [ ]:
NB = 300; rng = np.random.default_rng(11)
cells3 = [(a, ts, T) for a in A3 for ts, T in ALLC if (a, ts, NAT) in PR3]
PM3 = {c: pm(*c) for c in cells3}; BL = {c: np.zeros((NB, L)) for c in cells3}
for r in tqdm(range(NB), desc="bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}                      # one resample per site, shared by all architectures = paired
    for (a, ts, T) in cells3: BL[(a, ts, T)][r] = lab_auc(YT[T][ix[T]], PM3[(a, ts, T)][ix[T]], VALID[T])
BM = {c: np.nanmean(v, 1) for c, v in BL.items()}
PTL3 = {c: lab_auc(YT[c[2]], PM3[c], VALID[c[2]]) for c in cells3}; PTM3 = {c: np.nanmean(v) for c, v in PTL3.items()}
pci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)

GP3, GB3 = {}, {}
for a in A3:
    if not all(c in BL for c in [(a, "A", "A"), (a, "A", "B"), (a, "B", "B"), (a, "B", "A")]): print("skipped (missing models):", a); continue
    GP3[a] = dict(ab=PTL3[(a, "A", "A")] - PTL3[(a, "A", "B")], ba=PTL3[(a, "B", "B")] - PTL3[(a, "B", "A")])      # gap = in-hospital − external AUROC
    GB3[a] = dict(ab=BL[(a, "A", "A")] - BL[(a, "A", "B")], ba=BL[(a, "B", "B")] - BL[(a, "B", "A")])
    GP3[a]["asym"] = GP3[a]["ab"] - GP3[a]["ba"]; GB3[a]["asym"] = GB3[a]["ab"] - GB3[a]["ba"]
QN = {"ab": "Gap A→B (NIH→CheXpert)", "ba": "Gap B→A (CheXpert→NIH)", "asym": "Asymmetry (A→B gap − B→A gap)"}
rows = []
for a in GP3:
    for k, nm in QN.items():
        b = np.nanmean(GB3[a][k], 1); rows.append(dict(arch=a, quantity=nm, estimate=np.nanmean(GP3[a][k]), lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5)))
GT = pd.DataFrame(rows); display(GT.round(3)); GT.to_csv(os.path.join(VT, "table_arch3_gaps.csv"), index=False)
rows = []; arch_pairs = [("ConvNeXt-T", "DenseNet121"), ("ViT-S", "DenseNet121"), ("ViT-S", "ConvNeXt-T")]
for b_, a_ in arch_pairs:
    if b_ in GP3 and a_ in GP3:
        d = np.nanmean(GB3[b_]["asym"], 1) - np.nanmean(GB3[a_]["asym"], 1)
        rows.append(dict(pair=f"{b_} − {a_}", diff_in_asymmetry=np.nanmean(GP3[b_]["asym"]) - np.nanmean(GP3[a_]["asym"]), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5)))
DA = pd.DataFrame(rows); DA["CI_includes_0"] = (DA.lo <= 0) & (DA.hi >= 0); display(DA.round(3)); DA.to_csv(os.path.join(VT, "table_arch3_asymmetry_difference.csv"), index=False)

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.9), gridspec_kw=dict(width_ratios=[1, 1.5])); w = .26
for i, a in enumerate(GP3):
    g = GT[GT.arch == a]; ax[0].bar(np.arange(3) + (i - 1) * w, g.estimate, w, yerr=[np.clip(g.estimate - g.lo, 0, None), np.clip(g.hi - g.estimate, 0, None)], capsize=2, color=AC[a], edgecolor="k", lw=.4, label=a)
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(["Gap A→B", "Gap B→A", "Asymmetry"]); ax[0].axhline(0, color="k", lw=.6); ax[0].set(ylabel="Mean AUROC difference"); ax[0].legend(frameon=False); ax[0].grid(True, axis="y", alpha=.3)
x = np.arange(L)
for i, a in enumerate(GP3):
    for o, k, hatch in [(-.04, "ab", ""), (.04, "ba", "///")]:
        pt_ = GP3[a][k]; c = pci(GB3[a][k]); ax[1].bar(x + (i - 1) * .27 + o * 2.2, pt_, .12, yerr=[np.clip(pt_ - c[0], 0, None), np.clip(c[1] - pt_, 0, None)], capsize=1, color=AC[a], hatch=hatch, edgecolor="k", lw=.3)
ax[1].set_xticks(x); ax[1].set_xticklabels(LABELS6, rotation=35, ha="right"); ax[1].axhline(0, color="k", lw=.6); ax[1].set(ylabel="Per-label AUROC drop"); ax[1].set_title("Per label (solid: A→B, hatched: B→A)", fontsize=7); ax[1].grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_arch3_gaps_asymmetry")

In [ ]:
rows = []
for b_, a_ in arch_pairs:
    for ts, T in ALLC:
        if (b_, ts, T) in BM and (a_, ts, T) in BM:
            d = BM[(b_, ts, T)] - BM[(a_, ts, T)]
            rows.append(dict(pair=f"{b_} − {a_}", direction=dname(ts, T), seen=T in ts.split("+"), diff=PTM3[(b_, ts, T)] - PTM3[(a_, ts, T)], lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / NB)))
PD3 = pd.DataFrame(rows); o = np.argsort(PD3.p.values); holm = np.empty(len(PD3)); run = 0
for r_, i in enumerate(o): run = max(run, min(1, (len(PD3) - r_) * PD3.p.values[i])); holm[i] = run
PD3["p_holm"] = holm; PD3["significant"] = PD3.p_holm < .05; display(PD3.round(4)); PD3.to_csv(os.path.join(VT, "table_arch3_pairwise_differences.csv"), index=False)

dirs = [dname(ts, T) for ts, T in ALLC]; yy = {d: i for i, d in enumerate(dirs)}; mk = {"ConvNeXt-T − DenseNet121": "o", "ViT-S − DenseNet121": "s", "ViT-S − ConvNeXt-T": "^"}
fig, ax = plt.subplots(figsize=(COL2, 4.6))
for k, (pair, g) in enumerate(PD3.groupby("pair")):
    for _, r in g.iterrows():
        ax.errorbar(r["diff"], yy[r.direction] + (k - 1) * .22, xerr=[[r["diff"] - r.lo], [r.hi - r["diff"]]], fmt=mk[pair], color="k", mfc="k" if r.significant else "w", ms=3.5, capsize=1.5, lw=.7, label=pair if r.direction == dirs[0] else None)
ax.axvline(0, color="k", ls="--", lw=.6); ax.axhline(len(EXT) - .5, color="0.6", lw=.6); ax.set_yticks(range(len(dirs))); ax.set_yticklabels([d + ("  (seen)" if i >= len(EXT) else "") for i, d in enumerate(dirs)]); ax.invert_yaxis()
ax.set(xlabel="AUROC difference (95% CI; filled = significant after Holm)"); ax.legend(frameon=False, fontsize=6, loc="lower right"); ax.grid(True, axis="x", alpha=.3); fig.tight_layout(); save_fig(fig, "fig_arch3_pairwise_forest")

fig, ax = plt.subplots(figsize=(COL2, 3.0)); w = .26
for i, a in enumerate(A3):
    v = np.array([PTM3.get((a, ts, T), np.nan) for ts, T in EXT]); c = np.array([pci(BM[(a, ts, T)]) if (a, ts, T) in BM else [np.nan, np.nan] for ts, T in EXT])
    ax.bar(np.arange(len(EXT)) + (i - 1) * w, v, w, yerr=[np.clip(v - c[:, 0], 0, None), np.clip(c[:, 1] - v, 0, None)], capsize=1, color=AC[a], edgecolor="k", lw=.3, label=a)
ax.set_xticks(range(len(EXT))); ax.set_xticklabels([dname(*e) for e in EXT], rotation=40, ha="right"); ax.set_ylim(.55, None); ax.set(ylabel="External macro AUROC"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_arch3_external_auroc")

In [ ]:
rows = []
for a in A3:
    for ts, T in ALLC:
        if (a, ts, NAT) not in PR3: continue
        v = np.where(VALID[T])[0]; Y = YT[T]
        for s, d in PR3[(a, ts, NAT)].items():
            P = d[T]; rows.append(dict(arch=a, train=ts, test=T, seen=T in ts.split("+"), seed=s, ece=np.mean([ece_of(Y[:, j], P[:, j]) for j in v]), brier=np.mean((P[:, v] - Y[:, v]) ** 2)))
CA = pd.DataFrame(rows); CA.to_csv(os.path.join(VT, "table_arch3_calibration.csv"), index=False)
for m in ["ece", "brier"]:
    t = CA.groupby(["train", "test", "arch"])[m].agg(lambda x: f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}").unstack("arch"); print(f"\n=== {m.upper()} (lower is better) ==="); display(t)
fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.0)); w = .26
for a_, m, t in zip(ax, ["ece", "brier"], ["ECE (external directions)", "Brier score (external directions)"]):
    for i, a in enumerate(A3):
        g = [CA[(CA.arch == a) & (CA.train == ts) & (CA.test == T)][m] for ts, T in EXT]
        a_.bar(np.arange(len(EXT)) + (i - 1) * w, [x.mean() if len(x) else np.nan for x in g], w, yerr=[x.std(ddof=1) if len(x) > 1 else 0 for x in g], capsize=1, color=AC[a], edgecolor="k", lw=.3, label=a)
    a_.set_xticks(range(len(EXT))); a_.set_xticklabels([dname(*e) for e in EXT], rotation=40, ha="right"); a_.set_title(t); a_.grid(True, axis="y", alpha=.3)
ax[0].legend(frameon=False); fig.tight_layout(); save_fig(fig, "fig_arch3_calibration_bars")

fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.7), sharey=True)
for a_, (ts, T) in zip(axes, [("A", "B"), ("B", "A"), ("A+B", "C")]):
    v = np.where(VALID[T])[0]
    for a in A3:
        if (a, ts, NAT) in PR3: xs_, ys_ = reliab(YT[T][:, v].ravel(), pm(a, ts, T)[:, v].ravel()); a_.plot(xs_, ys_, ls=ALS[a], marker="o", ms=2.5, color=AC[a] if a != "ViT-S" else "0.5", label=a)
    a_.plot([0, 1], [0, 1], "k:", lw=.6); a_.set(title=dname(ts, T), xlabel="Predicted probability"); a_.grid(True, alpha=.3)
axes[0].set_ylabel("Observed frequency"); axes[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_arch3_reliability")

In [ ]:
def boot_dg(a, nb=400, seed=21):
    rng = np.random.default_rng(seed); items = [(ts, me) for ts in POOLED for me in METHS if (a, ts, me) in PR3]
    if (a, "A+B", NAT) not in PR3: return pd.DataFrame()
    PS = {k: pm(a, k[0], UNSEEN[k[0]], k[1]) for k in items}; B = {k: np.zeros(nb) for k in items}
    for r in range(nb):
        ix = {T: resample(T, rng) for T in "ABC"}
        for (ts, me), P in PS.items(): T = UNSEEN[ts]; B[(ts, me)][r] = mauc(YT[T][ix[T]], P[ix[T]], VALID[T])
    rows = []
    for ts in POOLED:
        if (ts, NAT) not in PS: continue
        for me in METHS[1:]:
            if (ts, me) not in PS: continue
            T = UNSEEN[ts]; d = B[(ts, me)] - B[(ts, NAT)]
            rows.append(dict(arch=a, train=ts, method=me, diff=mauc(YT[T], PS[(ts, me)], VALID[T]) - mauc(YT[T], PS[(ts, NAT)], VALID[T]), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5), p=min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / nb)))
    df = pd.DataFrame(rows)
    if len(df):
        o = np.argsort(df.p.values); holm = np.empty(len(df)); run = 0
        for r_, i in enumerate(o): run = max(run, min(1, (len(df) - r_) * df.p.values[i])); holm[i] = run
        df["p_holm"] = holm; df["significant"] = df.p_holm < .05
    return df
DGA = pd.concat([boot_dg(a) for a in A3]); display(DGA.round(4)); DGA.to_csv(os.path.join(VT, "table_arch_dg_vs_erm.csv"), index=False)

rows = []
for a in A3:
    for ts in POOLED:
        for me in METHS:
            if (a, ts, me) not in PR3: continue
            T = UNSEEN[ts]; v = [mauc(YT[T], PR3[(a, ts, me)][s][T], VALID[T]) for s in PR3[(a, ts, me)]]
            rows.append(dict(arch=a, train=ts, unseen=T, method=me, seeds=len(v), auroc=np.mean(v), sd=np.std(v, ddof=1) if len(v) > 1 else 0))
DG3 = pd.DataFrame(rows); DG3.to_csv(os.path.join(VT, "table_arch_dg_methods.csv"), index=False)
display(DG3.assign(v=DG3.apply(lambda r: f"{r.auroc:.3f} ± {r.sd:.3f}", axis=1)).pivot(index="method", columns=["train", "arch"], values="v").reindex(METHS))

fig, axes = plt.subplots(1, 3, figsize=(COL2, 3.0), sharey=True); w = .26
for a_, ts in zip(axes, SETS_ORDER):
    for i, a in enumerate(A3):
        g = DG3[(DG3.arch == a) & (DG3.train == ts)].set_index("method").reindex(METHS)
        a_.bar(np.arange(len(METHS)) + (i - 1) * w, g.auroc, w, yerr=g.sd, capsize=1, color=AC[a], edgecolor="k", lw=.3, label=a)
    a_.set_xticks(range(len(METHS))); a_.set_xticklabels(METHS, rotation=40, ha="right", fontsize=6); a_.set_title(f"{ts}→{UNSEEN[ts]}"); a_.set_ylim(.6, None); a_.grid(True, axis="y", alpha=.3)
axes[0].set_ylabel("Unseen-site macro AUROC"); axes[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_arch3_dg_methods")

MKR = {"DenseNet121": "o", "ConvNeXt-T": "^", "ViT-S": "s"}; MROW = METHS[1:]
fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.9), sharex=True)
for a_, ts in zip(axes, SETS_ORDER):
    for i, a in enumerate(A3):
        if not len(DGA): break
        for _, r in DGA[(DGA.arch == a) & (DGA.train == ts)].iterrows():
            a_.errorbar(r["diff"], MROW.index(r.method) + (i - 1) * .25, xerr=[[r["diff"] - r.lo], [r.hi - r["diff"]]], fmt=MKR[a], color="k", mfc="k" if r.significant else "w", ms=4, capsize=1.5, lw=.7, label=a if r.method == MROW[0] else None)
    a_.axvline(0, color="k", ls="--", lw=.6); a_.set_yticks(range(len(MROW))); a_.set_yticklabels(MROW if a_ is axes[0] else [""] * len(MROW)); a_.invert_yaxis()
    a_.set(title=f"{ts}→{UNSEEN[ts]}", xlabel="AUROC − ERM (natural)"); a_.grid(True, axis="x", alpha=.3)
axes[0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_arch3_dg_forest")

In [ ]:
from itertools import combinations
def ens(ts, T, names): return np.mean([pm(a, ts, T) for a in names], 0)
combos = {"DenseNet121": ["DenseNet121"], "ConvNeXt-T": ["ConvNeXt-T"], "ViT-S": ["ViT-S"], "D + C": ["DenseNet121", "ConvNeXt-T"], "D + V": ["DenseNet121", "ViT-S"], "C + V": ["ConvNeXt-T", "ViT-S"], "D + C + V": A3}
okc = [(ts, T) for ts, T in EXT if all((a, ts, NAT) in PR3 for a in A3)]
rows = [dict(direction=dname(ts, T), ensemble=k, auroc=mauc(YT[T], ens(ts, T, v), VALID[T])) for ts, T in okc for k, v in combos.items()]
EN = pd.DataFrame(rows); tab = EN.pivot(index="direction", columns="ensemble", values="auroc")[list(combos)]
tab["gain over best single"] = tab["D + C + V"] - tab[["DenseNet121", "ConvNeXt-T", "ViT-S"]].max(1); display(tab.round(4)); tab.to_csv(os.path.join(VT, "table_arch3_ensembles.csv"))

rng = np.random.default_rng(13); NBE = 200; KEY = ["DenseNet121", "ConvNeXt-T", "ViT-S", "D + C + V"]; BE = {(dname(*c), k): np.zeros(NBE) for c in okc for k in KEY}
for r in tqdm(range(NBE), desc="ensemble bootstrap"):
    ix = {T: resample(T, rng) for T in "ABC"}
    for ts, T in okc:
        for k in KEY: BE[(dname(ts, T), k)][r] = mauc(YT[T][ix[T]], ens(ts, T, combos[k])[ix[T]], VALID[T])
fig, ax = plt.subplots(figsize=(COL2, 3.0)); w = .2; cols = ["0.2", "0.55", "0.85", "0.98"]
for i, k in enumerate(KEY):
    v = np.array([tab.loc[dname(*c), k] for c in okc]); cc = np.array([pci(BE[(dname(*c), k)]) for c in okc])
    ax.bar(np.arange(len(okc)) + (i - 1.5) * w, v, w, yerr=[np.clip(v - cc[:, 0], 0, None), np.clip(cc[:, 1] - v, 0, None)], capsize=1, color=cols[i], edgecolor="k", lw=.4, hatch="///" if i == 3 else "", label=k)
ax.set_xticks(range(len(okc))); ax.set_xticklabels([dname(*c) for c in okc], rotation=40, ha="right"); ax.set_ylim(.55, None); ax.set(ylabel="Macro AUROC"); ax.legend(frameon=False, ncol=2, fontsize=6); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_arch3_ensembles")

def rank_corr(P1, P2, T):                                           # mean Spearman correlation of scores over valid labels
    v = np.where(VALID[T])[0]; return np.mean([np.corrcoef(rankdata(P1[:, j]), rankdata(P2[:, j]))[0, 1] for j in v])
rows = [dict(direction=dname(ts, T), pair=f"{a} vs {b}", spearman=rank_corr(pm(a, ts, T), pm(b, ts, T), T)) for ts, T in okc for a, b in combinations(A3, 2)]
DV = pd.DataFrame(rows).pivot(index="direction", columns="pair", values="spearman"); display(DV.round(3)); DV.to_csv(os.path.join(VT, "table_arch3_diversity.csv"))
fig, ax = plt.subplots(figsize=(COL1 * 1.6, 3.2)); im = ax.imshow(DV.values, cmap="viridis", vmin=DV.values.min() - .02, vmax=1)
for i in range(DV.shape[0]):
    for j in range(DV.shape[1]): ax.text(j, i, f"{DV.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=6)
ax.set_xticks(range(DV.shape[1])); ax.set_xticklabels([c.replace(" vs ", "\nvs\n") for c in DV.columns], fontsize=5.5); ax.set_yticks(range(DV.shape[0])); ax.set_yticklabels(DV.index); ax.grid(False)
fig.colorbar(im, ax=ax, label="Spearman correlation of scores"); ax.set_title("Score agreement between architectures"); fig.tight_layout(); save_fig(fig, "fig_arch3_diversity")

In [ ]:
rows = []; HP = {"DenseNet121": os.path.join(OUT, "nih_train", "densenet121_nih_e7_seed0_history.csv"),
                 "ConvNeXt-T": os.path.join(OUT, "part34", "convnext_tiny_nih_e7_seed0_history.csv"), "ViT-S": base_path("A", 0) + "_history.csv"}
for a, n in NAMEM.items():
    m = timm.create_model(n, pretrained=False, num_classes=L).to(dev); npar = sum(p.numel() for p in m.parameters()) / 1e6
    m.eval(); x = torch.randn(128, 3, 224, 224, device=dev)
    with torch.inference_mode(), torch.autocast("cuda"):
        for _ in range(5): m(x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(20): m(x)
        torch.cuda.synchronize(); ips = 20 * 128 / (time.time() - t0)
    m.train(); torch.cuda.reset_peak_memory_stats(); opt = torch.optim.AdamW(m.parameters(), 1e-4); xb_ = torch.randn(BS, 3, 224, 224, device=dev)
    with torch.autocast("cuda"): loss = m(xb_).float().mean()
    loss.backward(); opt.step(); torch.cuda.synchronize(); mem = torch.cuda.max_memory_allocated() / 1e9
    try: h = pd.read_csv(HP[a]); ep_min = h["sec"].mean() / 60 if "sec" in h else np.nan
    except Exception: ep_min = np.nan
    rows.append({"architecture": a, "params (M)": round(npar, 1), "inference img/s (fp16, bs128)": round(ips), f"train step peak GB (bs{BS})": round(mem, 1), "min/epoch on NIH (from your logs)": round(ep_min, 1) if not np.isnan(ep_min) else "n/a"})
    del m, opt, xb_, x, loss; gc.collect(); torch.cuda.empty_cache()
EF = pd.DataFrame(rows); display(EF); EF.to_csv(os.path.join(VT, "table_arch3_efficiency.csv"), index=False)

In [ ]:
from sklearn.manifold import TSNE
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
CK_AB = {"DenseNet121": os.path.join(OUT, "dg", "erm_nat_e7_seed0.pt"), "ConvNeXt-T": os.path.join(OUT, "c_source", "convnext_ApB_erm_nat_e7_seed0.pt"), "ViT-S": base_path("A+B", 0) + ".pt"}
AV35 = [a for a in A3 if os.path.exists(CK_AB[a])]
for a in A3:
    if a not in AV35: print(f"SKIPPED {a}: checkpoint not found -> {CK_AB[a]}   (edit CK_AB with the real path of your A+B ERM natural seed-0 checkpoint)")
IMN = (torch.tensor([.485, .456, .406], device=dev).view(1, 3, 1, 1), torch.tensor([.229, .224, .225], device=dev).view(1, 3, 1, 1))
NORM = {"DenseNet121": IMN, "ConvNeXt-T": IMN, "ViT-S": (MEAN, STD)}
def load_arch(a, path):
    m = timm.create_model(NAMEM[a], pretrained=False, num_classes=L).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=True)); return m.eval()
def xb(A, idx, a):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1); return (x - NORM[a][0]) / NORM[a][1]
@torch.inference_mode()
def emb(m, a, A, idx):
    out = []
    for i in range(0, len(idx), 100):
        with torch.autocast("cuda"): out.append(m.forward_head(m.forward_features(xb(A, idx[i:i + 100], a)), pre_logits=True).float().cpu())
    return torch.cat(out).numpy()
rng = np.random.default_rng(0); SEL = {T: np.sort(rng.choice(len(TEST[T][1]), 600, replace=False)) for T in "ABC"}; site = np.repeat(list("ABC"), 600)
EMB = {}
for a in AV35:
    m = load_arch(a, CK_AB[a]); EMB[a] = np.concatenate([emb(m, a, TEST[T][0], TEST[T][1][SEL[T]]) for T in "ABC"]); del m; torch.cuda.empty_cache()
raw = np.concatenate([F.adaptive_avg_pool2d(torch.from_numpy(np.ascontiguousarray(TEST[T][0][TEST[T][1][SEL[T]]])).float()[:, None], 28).flatten(1).numpy() for T in "ABC"])
probe = lambda X: cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X, site, cv=5).mean()
PB = {**{a: probe(EMB[a]) for a in AV35}, "raw pixels (28×28)": probe(raw)}
display(pd.Series(PB, name="site-probe accuracy (chance 0.33)").round(3).to_frame()); pd.Series(PB).to_csv(os.path.join(VT, "table_arch3_site_probe.csv"))
fig, axes = plt.subplots(1, len(AV35), figsize=(COL2 * len(AV35) / 3, 2.9), squeeze=False); axes = axes[0]
for a_, a in zip(axes, AV35):
    Z2 = TSNE(2, perplexity=35, random_state=0, init="pca").fit_transform(EMB[a])
    for T, c in zip("ABC", ["C0", "C1", "C2"]): a_.scatter(*Z2[site == T].T, s=3, alpha=.6, color=c, label=SITE[T])
    a_.set(title=f"{a}\nprobe acc {PB[a]:.2f}", xticks=[], yticks=[])
axes[0].legend(frameon=False, markerscale=3, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_arch3_tsne_site")

In [ ]:
PA_ = {"DenseNet121": os.path.join(OUT, "nih_train", "densenet121_nih_e7_seed0.pt"), "ConvNeXt-T": os.path.join(OUT, "part34", "convnext_tiny_nih_e7_seed0.pt"), "ViT-S": base_path("A", 0) + ".pt"}
AV36 = [a for a in A3 if os.path.exists(PA_[a])]
for a in A3:
    if a not in AV36: print(f"SKIPPED {a}: checkpoint not found -> {PA_[a]}   (edit PA_ with the real path of your NIH-trained seed-0 checkpoint)")
def make_cam(m, a):
    if a == "ViT-S": vc = ViTCAM(m); return lambda x, j: vc(x, j)[0]
    def f(x, j):
        with torch.enable_grad():
            fm = m.forward_features(x); out = m.forward_head(fm); g = torch.autograd.grad(out[:, j].sum(), fm)[0]
        cam = torch.relu((g.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
        return F.interpolate(cam, size=224, mode="bilinear", align_corners=False)[:, 0].detach().cpu().numpy()
    return f
nrm = lambda c: c / (c.reshape(len(c), -1).max(1)[:, None, None] + 1e-8)
def bp(ax, data, names):                                                 # boxplot that works on old and new Matplotlib
    try: ax.boxplot(data, tick_labels=names, showfliers=False)
    except TypeError: ax.boxplot(data, labels=names, showfliers=False)
LAB = "Pleural Effusion"; jl = LABELS6.index(LAB); pos = np.where(YT["B"][:, jl] == 1)[0]; rng = np.random.default_rng(7); NST = 150; idx_st = np.sort(rng.choice(pos, NST, replace=False)); show = idx_st[:6]
CAMS = {}
for a in AV36:
    m = load_arch(a, PA_[a]); cam = make_cam(m, a); out = []
    for i in range(0, NST, 25): out.append(cam(xb(CX_T, idx_st[i:i + 25], a), jl))
    CAMS[a] = nrm(np.concatenate(out)); del m, cam; torch.cuda.empty_cache()

fig, ax = plt.subplots(len(AV36), 6, figsize=(COL2, 1.3 * len(AV36) + .3), squeeze=False)
for r_, a in enumerate(AV36):
    for c_ in range(6): ax[r_, c_].imshow(CX_T[show[c_]], cmap="gray"); ax[r_, c_].imshow(CAMS[a][c_], cmap="jet", alpha=.4); ax[r_, c_].axis("off")
    ax[r_, 0].set_title(a, fontsize=7, loc="left")
fig.suptitle(f"Grad-CAM, {LAB}: NIH-trained models on CheXpert positives (same images in every row)", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_arch3_gradcam_compare")

def bshare(c, f=.1):
    k = int(224 * f); mk_ = np.ones((224, 224), bool); mk_[k:-k, k:-k] = False; return c[:, mk_].sum(1) / (c.reshape(len(c), -1).sum(1) + 1e-8)
down = lambda c: F.avg_pool2d(torch.from_numpy(c)[:, None], 4)[:, 0].numpy().reshape(len(c), -1)
def top_iou(c1, c2, q=80):
    a_, b_ = c1 >= np.percentile(c1, q, axis=1, keepdims=True), c2 >= np.percentile(c2, q, axis=1, keepdims=True); return (a_ & b_).sum(1) / ((a_ | b_).sum(1) + 1e-8)
BR3 = pd.DataFrame([dict(arch=a, border_share=x) for a in AV36 for x in bshare(CAMS[a])]); print("uniform CAM = 0.36 in the outer 10% frame"); display(BR3.groupby("arch").border_share.agg(["mean", "std"]).round(3))
rows = []
for a, b in combinations(AV36, 2):
    da, db = down(CAMS[a]), down(CAMS[b]); pear = np.array([np.corrcoef(x, y)[0, 1] for x, y in zip(da, db)]); iou = top_iou(da, db)
    rows += [dict(pair=f"{a} vs {b}", metric="Pearson r", value=v) for v in pear] + [dict(pair=f"{a} vs {b}", metric="top-20% IoU", value=v) for v in iou]
AG = pd.DataFrame(rows); display(AG.groupby(["pair", "metric"]).value.agg(["mean", "std"]).round(3)); AG.to_csv(os.path.join(VT, "table_arch3_gradcam_agreement.csv"), index=False)
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.7))
bp(ax[0], [BR3[BR3.arch == a].border_share for a in AV36], [a.replace("-", "-\n") for a in AV36]); ax[0].axhline(.36, color="r", ls="--", lw=.7); ax[0].set(ylabel="CAM mass in outer 10% border", title="Border share")
for a_, met in zip(ax[1:], ["Pearson r", "top-20% IoU"]):
    g = AG[AG.metric == met]; ps_ = list(g.pair.unique()); bp(a_, [g[g.pair == p].value for p in ps_], [p.replace(" vs ", "\nvs\n") for p in ps_]); a_.set(title=f"Agreement ({met})")
for a_ in ax: a_.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_arch3_gradcam_stats")

In [ ]:
print("=== key findings to check against the tables ===")
for a in GP3: g = GT[(GT.arch == a) & GT.quantity.str.startswith("Asym")].iloc[0]; print(f"{a:12s} asymmetry {g.estimate:+.3f}  [{g.lo:+.3f}, {g.hi:+.3f}]  excludes 0: {not (g.lo <= 0 <= g.hi)}")
print("\nasymmetry differs between architectures? (CI includes 0 = NOT architecture-specific)"); display(DA.round(3))
print("\nsignificant architecture differences on EXTERNAL directions (Holm):"); display(PD3[PD3.significant & ~PD3.seen][["pair", "direction", "diff", "lo", "hi", "p_holm"]].round(4))
print("\nDG methods that beat ERM (natural), per architecture:"); display(DGA[(DGA.significant) & (DGA["diff"] > 0)] if len(DGA) else DGA)
import shutil; shutil.make_archive(os.path.join(OUT, "ALL_bundle"), "zip", VT)
print("figures:", sorted(f for f in os.listdir(FIG) if f.endswith(".png")))